# Model compression for skin lesion classification: rare-class recall and robustness

Code for the ISBI 2027 submission. Run the parts in order on a Colab GPU (T4). See README.md.

**Part 0** setup and data. **Part 1** first run (fold 0 only, kept for transparency; it was used to see that the first corruption levels were too strong; not used for the paper's tables). **Part 2** seven-fold cross-validation. **Part 3** analysis, tables, figure.

## Part 0. Setup and data

In [ ]:
#cell 1
import os, sys, time, random, glob, io, shutil
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/RareClassRecall_SkinLesion'   # <- your Drive folder
os.makedirs(PROJECT_DIR, exist_ok=True)

import torch, torchvision, numpy, pandas, sklearn, PIL
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scikit-learn", sklearn.__version__, "| Pillow", PIL.__version__)
print("GPU available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("CPU cores:", os.cpu_count())

# check that we can write to the Drive folder
test_file = os.path.join(PROJECT_DIR, "_write_test.txt")
open(test_file, "w").write("ok"); os.remove(test_file)
print("Can write to the Drive folder: OK")

# check that the input files exist
for name in ["HAM10000_metadata.csv", "ham10000_images_part_1", "ham10000_images_part_2"]:
    p = os.path.join(PROJECT_DIR, name)
    extra = ""
    if os.path.isdir(p):
        extra = f"({len(os.listdir(p))} files)"
    print(f"{name:28s} ->", "found" if os.path.exists(p) else "NOT FOUND", extra)

In [ ]:
#cell 2
import os, sys, time, random, glob, io, shutil
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/RareClassRecall_SkinLesion'
os.makedirs(PROJECT_DIR, exist_ok=True)

import torch, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

META_CSV  = os.path.join(PROJECT_DIR, "HAM10000_metadata.csv")
SPLIT_CSV = os.path.join(PROJECT_DIR, "ham10000_lesion_split.csv")

df = pd.read_csv(META_CSV).reset_index(drop=True)
need = ["lesion_id", "image_id", "dx"]
assert all(c in df.columns for c in need), "metadata columns missing"

missing = df[need].isnull().sum()
print("\nMissing values in the columns we use:\n", missing)
assert missing.sum() == 0, "missing values found"
assert not df["image_id"].duplicated().any(), "duplicate image_id rows"
assert set(df["dx"].unique()) == {"akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"}, "unexpected class names"

print(f"\n{len(df)} images, {df['lesion_id'].nunique()} lesions")
conf = int((df.groupby("lesion_id")["dx"].nunique() > 1).sum())
print("Lesions with conflicting labels:", conf, "(must be 0)")

if os.path.exists(SPLIT_CSV):
    old = pd.read_csv(SPLIT_CSV)
    assert len(old) == len(df) and (old["image_id"].values == df["image_id"].values).all(), \
        "existing split file does not match the metadata file"
    df["split"] = old["split"].values
    print("\nExisting split file found and reused:", SPLIT_CSV)
else:
    # 7 folds: fold 0 -> test (~14%), fold 1 -> val (~14%), the other 5 -> train (~71%)
    sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=0)
    folds = [t for _, t in sgkf.split(df, df["dx"], groups=df["lesion_id"])]
    df["split"] = "train"
    df.loc[folds[0], "split"] = "test"
    df.loc[folds[1], "split"] = "val"
    df.to_csv(SPLIT_CSV, index=False)
    print("\nNew split saved to:", SPLIT_CSV)

leaked = int((df.groupby("lesion_id")["split"].nunique() > 1).sum())
print("Lesions appearing in more than one split:", leaked, "(must be 0)")
assert leaked == 0, "LEAKAGE!"

table = pd.crosstab(df["dx"], df["split"])[["train", "val", "test"]]
print("\nImages per class per split:")
print(table)
print("\n", df["split"].value_counts())
print("\nOur earlier Kaggle run gave: train 7176, val 1431, test 1408; df test 12, vasc test 13.")
print("A small difference is OK. What matters: leakage = 0 and the SAME split file for everything.")

In [ ]:
#cell 3
import numpy as np
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

SIZE = 128
NPZ_DRIVE = os.path.join(PROJECT_DIR, f"ham10000_{SIZE}.npz")
NPZ_LOCAL = f"/content/ham10000_{SIZE}.npz"

if os.path.exists(NPZ_DRIVE):
    print("Cache file already exists on Drive. Copying it to the fast local disk...")
    shutil.copy(NPZ_DRIVE, NPZ_LOCAL)
    print("Done.")
else:
    folders = [os.path.join(PROJECT_DIR, "ham10000_images_part_1"),
               os.path.join(PROJECT_DIR, "ham10000_images_part_2")]
    path_of = {}                      # image_id -> file path (only these two folders are searched)
    for fol in folders:
        assert os.path.isdir(fol), f"folder not found: {fol}"
        for f in os.listdir(fol):
            stem, ext = os.path.splitext(f)
            if ext.lower() in (".jpg", ".jpeg", ".png"):
                path_of.setdefault(stem, os.path.join(fol, f))
    print("Unique image files found:", len(path_of), "(must be 10015)")

    ids = df["image_id"].tolist()
    miss = [i for i in ids if i not in path_of]
    assert not miss, f"{len(miss)} images missing, for example {miss[:5]}"

    class_names = sorted(df["dx"].unique())          # alphabetical: akiec, bcc, bkl, df, mel, nv, vasc
    c2i = {c: i for i, c in enumerate(class_names)}
    resample = getattr(Image, "Resampling", Image).BILINEAR
    X = np.zeros((len(ids), SIZE, SIZE, 3), dtype=np.uint8)

    def load_one(i):
        img = Image.open(path_of[ids[i]]).convert("RGB").resize((SIZE, SIZE), resample)
        X[i] = np.asarray(img, dtype=np.uint8)

    t0 = time.time(); done = 0
    with ThreadPoolExecutor(max_workers=8) as pool:
        for _ in pool.map(load_one, range(len(ids))):
            done += 1
            if done % 1000 == 0:
                print(f"  {done}/{len(ids)} images  ({time.time() - t0:.0f} s)")
    print(f"Loaded and resized in {time.time() - t0:.0f} s")

    # ---- sanity checks ----
    print("X shape:", X.shape, "| dtype:", X.dtype)
    print("Pixel mean (0-255):", round(float(X.mean()), 1), "(should be roughly 100-180, not 0 or 255)")
    print("Channel means R,G,B:", X.mean(axis=(0, 1, 2)).round(1))
    stds = np.array([X[i].astype(np.float32).std() for i in range(len(X))])
    print("Almost blank images (std < 2):", int((stds < 2).sum()), "(expect 0)")

    y = df["dx"].map(c2i).to_numpy(dtype=np.int64)
    np.savez_compressed(
        NPZ_LOCAL, X=X, y=y,
        split=np.array(df["split"].tolist(), dtype=str),
        lesion=np.array(df["lesion_id"].tolist(), dtype=str),
        image_id=np.array(ids, dtype=str),
        class_names=np.array(class_names, dtype=str))
    shutil.copy(NPZ_LOCAL, NPZ_DRIVE)
    print(f"Saved: {NPZ_DRIVE}  ({os.path.getsize(NPZ_DRIVE) / 1e6:.0f} MB)")

## Part 1. First run (fold 0, three seeds; kept for transparency)
This part is not needed to reproduce the paper's tables. Skip it if you only want the final results.

In [ ]:
#cell 4
import os, time, random, io, shutil, contextlib
import numpy as np
import torch, torchvision
import torch.nn as nn
import torch.nn.functional as F

SIZE = 128

# ------------------------------ settings ------------------------------
SMOKE = False
SEEDS = [0, 1, 2]          # True = tiny test run (1-3 min). Set to False for the real run. SEEDS = [0, 1, 2]
EPOCHS = 15
BATCH = 64
LR = 3e-4
WEIGHT_DECAY = 1e-4
TAIL_CLASSES = ["akiec", "df", "vasc"]     # fixed in advance
MEL_CLASS = "mel"
# Plain cross-entropy, NO class weights, NO focal loss, NO oversampling (fixed decision).

RUN_DIR  = os.path.join(PROJECT_DIR, "smoke_run" if SMOKE else "run")
CKPT_DIR = os.path.join(RUN_DIR, "ckpt")
PRED_DIR = os.path.join(RUN_DIR, "preds")
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(PRED_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cpu" and not SMOKE:
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU. (Or use SMOKE = True.)")

MEAN = (0.485, 0.456, 0.406)    # fixed ImageNet numbers (not learned from our data)
STD = (0.229, 0.224, 0.225)


# ------------------------------ helpers ------------------------------
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def per_class_recall(y_true, y_pred, n_classes):
    rec = np.full(n_classes, np.nan)
    for c in range(n_classes):
        m = y_true == c
        if m.sum() > 0:
            rec[c] = float((y_pred[m] == c).mean())
    return rec

def summarize(y_true, y_pred, class_names):
    n = len(class_names)
    rec = per_class_recall(y_true, y_pred, n)
    idx = {c: i for i, c in enumerate(class_names)}
    tail_mask = np.isin(y_true, [idx[c] for c in TAIL_CLASSES])
    mel_mask = y_true == idx[MEL_CLASS]
    return {
        "accuracy": float((y_true == y_pred).mean()),
        "balanced_acc": float(np.nanmean(rec)),                 # macro recall
        "tail_recall": float((y_pred[tail_mask] == y_true[tail_mask]).mean()) if tail_mask.any() else float("nan"),
        "mel_recall": float((y_pred[mel_mask] == y_true[mel_mask]).mean()) if mel_mask.any() else float("nan"),
        "per_class_recall": {c: rec[i] for i, c in enumerate(class_names)},
    }

def autocast(dev, use_amp):
    if not use_amp:
        return contextlib.nullcontext()
    return torch.autocast(device_type=dev.type, dtype=torch.float16)

def make_scaler(use_amp):
    try:
        return torch.amp.GradScaler("cuda", enabled=use_amp)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=use_amp)

def prep(xb_uint8):
    """uint8 (N,H,W,3) tensor -> normalised float (N,3,H,W) tensor."""
    x = xb_uint8.permute(0, 3, 1, 2).float() / 255.0
    mean = torch.tensor(MEAN, device=x.device).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=x.device).view(1, 3, 1, 1)
    return (x - mean) / std

def augment(x):
    """TRAIN only: random horizontal flip, vertical flip (per image), 90-degree rotation (per batch)."""
    b = x.size(0)
    fh = torch.rand(b, device=x.device) < 0.5
    x = torch.where(fh[:, None, None, None], torch.flip(x, dims=[3]), x)
    fv = torch.rand(b, device=x.device) < 0.5
    x = torch.where(fv[:, None, None, None], torch.flip(x, dims=[2]), x)
    k = random.randint(0, 3)
    if k:
        x = torch.rot90(x, k, dims=[2, 3])
    return x

def build_resnet50(n_classes, pretrained=False):
    if pretrained:
        try:
            m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        except Exception as e:
            raise RuntimeError("Could not download the pre-trained ResNet-50 weights (is the internet on?). "
                               "Original error: " + str(e))
    else:
        m = torchvision.models.resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    return m

@torch.no_grad()
def predict_logits(model, X_np, dev, use_amp, bs=128):
    model.eval()
    outs = []
    for i in range(0, len(X_np), bs):
        xb = prep(torch.from_numpy(X_np[i:i + bs]).to(dev))
        with autocast(dev, use_amp):
            out = model(xb)
        outs.append(out.float().cpu())
    return torch.cat(outs).numpy()

def state_size_mb(model):
    buf = io.BytesIO(); torch.save(model.state_dict(), buf)
    return buf.tell() / 1e6

def count_nonzero_params(model):
    return int(sum((p != 0).sum().item() for p in model.parameters()))

def pick_smoke_subset(split, y, which, per_class):
    ids = np.where(split == which)[0]
    keep = []
    for c in np.unique(y[ids]):
        keep.extend(ids[y[ids] == c][:per_class].tolist())
    return np.array(sorted(keep))


# ------------------------------ load the cached data ------------------------------
NPZ_LOCAL = f"/content/ham10000_{SIZE}.npz"
NPZ_DRIVE = os.path.join(PROJECT_DIR, f"ham10000_{SIZE}.npz")
if not os.path.exists(NPZ_LOCAL):
    assert os.path.exists(NPZ_DRIVE), "Cache file not found. Run Cell 3 first."
    shutil.copy(NPZ_DRIVE, NPZ_LOCAL)
d = np.load(NPZ_LOCAL)
X, y, split, lesion = d["X"], d["y"], d["split"], d["lesion"]
class_names = [str(c) for c in d["class_names"]]
print("Classes:", class_names, "| images:", X.shape)

if SMOKE:
    tr = pick_smoke_subset(split, y, "train", 40)
    va = pick_smoke_subset(split, y, "val", 20)
    te = pick_smoke_subset(split, y, "test", 20)
    seeds_to_run, epochs_run, batch_run = [0], 1, 32
else:
    tr, va, te = (np.where(split == s)[0] for s in ("train", "val", "test"))
    seeds_to_run, epochs_run, batch_run = SEEDS, EPOCHS, BATCH
print(f"train {len(tr)} | val {len(va)} | test {len(te)} | seeds {seeds_to_run} | epochs {epochs_run} | SMOKE={SMOKE}")

In [ ]:

#cell 5
def train_baseline(seed):
    use_amp = device.type == "cuda"
    set_seed(seed)
    model = build_resnet50(len(class_names), pretrained=True).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs_run)
    scaler = make_scaler(use_amp)
    loss_fn = nn.CrossEntropyLoss()                    # plain, no class weights

    Xtr_d = torch.from_numpy(X[tr]).to(device)
    ytr_d = torch.from_numpy(y[tr]).to(device)
    n = len(Xtr_d)

    best_score, best_epoch, best_state = -1.0, -1, None
    for epoch in range(epochs_run):
        t0 = time.time()
        model.train()
        perm = torch.randperm(n, device=device)
        total = 0.0
        for i in range(0, n, batch_run):
            idx = perm[i:i + batch_run]
            if len(idx) < 2:
                continue
            xb = augment(prep(Xtr_d[idx]))
            yb = ytr_d[idx]
            with autocast(device, use_amp):
                loss = loss_fn(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total += loss.item() * len(idx)
        sched.step()

        va_logits = predict_logits(model, X[va], device, use_amp)
        va_loss = float(F.cross_entropy(torch.from_numpy(va_logits), torch.from_numpy(y[va])))
        s = summarize(y[va], va_logits.argmax(1), class_names)
        print(f"seed {seed} epoch {epoch + 1:2d}/{epochs_run}  train_loss {total / n:.3f}  val_loss {va_loss:.3f}  "
              f"val_acc {s['accuracy']:.3f}  val_bal_acc {s['balanced_acc']:.3f}  ({time.time() - t0:.0f}s)")
        if s["balanced_acc"] > best_score:          # choose the epoch on VALIDATION only
            best_score, best_epoch = s["balanced_acc"], epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    print(f"seed {seed}: best epoch = {best_epoch}  (val balanced accuracy {best_score:.3f})")
    torch.save(best_state, os.path.join(CKPT_DIR, f"baseline_seed{seed}.pt"))
    va_logits = predict_logits(model, X[va], device, use_amp)
    te_logits = predict_logits(model, X[te], device, use_amp)
    np.savez_compressed(os.path.join(PRED_DIR, f"baseline_seed{seed}.npz"),
                        val_logits=va_logits, test_logits=te_logits,
                        y_val=y[va], y_test=y[te], lesion_test=lesion[te])
    st = summarize(y[te], te_logits.argmax(1), class_names)      # sanity check ONLY, never tune on this
    print(f"seed {seed} TEST (sanity only): acc {st['accuracy']:.3f}  bal_acc {st['balanced_acc']:.3f}  "
          f"tail_recall {st['tail_recall']:.3f}  mel_recall {st['mel_recall']:.3f}")


for seed in seeds_to_run:
    done = (os.path.exists(os.path.join(CKPT_DIR, f"baseline_seed{seed}.pt")) and
            os.path.exists(os.path.join(PRED_DIR, f"baseline_seed{seed}.npz")))
    if done and not SMOKE:
        print(f"seed {seed}: already done, skipping")
        continue
    t0 = time.time()
    train_baseline(seed)
    print(f"seed {seed} finished in {(time.time() - t0) / 60:.1f} min\n")
print("DONE. Files are in", RUN_DIR)

In [ ]:
#cell 6
for seed in seeds_to_run:
    p = np.load(os.path.join(PRED_DIR, f"baseline_seed{seed}.npz"))
    s = summarize(p["y_val"], p["val_logits"].argmax(1), class_names)
    print(f"seed {seed} VALIDATION: accuracy {s['accuracy']:.3f} | balanced accuracy {s['balanced_acc']:.3f}")
    print("  recall per class:", {k: (None if v != v else round(v, 2)) for k, v in s["per_class_recall"].items()})
print("\nHow to read this: accuracy can look good because nv is ~67% of the data.")
print("Balanced accuracy and the recall of akiec / df / vasc / mel show how the rare classes do.")

In [ ]:
#cell 7
import torch.nn.utils.prune as prune

SPARSITY = 0.75
FT_EPOCHS = 5
FT_LR = 1e-4
ft_epochs_run = 1 if SMOKE else FT_EPOCHS
PRUNE_NAME = f"pruned{int(SPARSITY * 100)}"

def prunable(model):
    return [(m, "weight") for m in model.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]

def prune_and_finetune(base_state, seed):
    use_amp = device.type == "cuda"
    set_seed(seed)
    model = build_resnet50(len(class_names), pretrained=False)
    model.load_state_dict(base_state)
    model.to(device)

    # global L1 magnitude pruning: the smallest 75% of ALL conv/linear weights (all layers together) -> 0
    prune.global_unstructured(prunable(model), pruning_method=prune.L1Unstructured, amount=SPARSITY)

    opt = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=WEIGHT_DECAY)   # created AFTER pruning
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ft_epochs_run)
    scaler = make_scaler(use_amp)
    loss_fn = nn.CrossEntropyLoss()
    Xtr_d = torch.from_numpy(X[tr]).to(device); ytr_d = torch.from_numpy(y[tr]).to(device)
    n = len(Xtr_d)

    s0 = summarize(y[va], predict_logits(model, X[va], device, use_amp).argmax(1), class_names)
    print(f"  right after pruning, BEFORE fine-tuning: val_bal_acc {s0['balanced_acc']:.3f}  val_acc {s0['accuracy']:.3f}")

    best_score, best_epoch, best_state = -1.0, -1, None
    for epoch in range(ft_epochs_run):
        t0 = time.time(); model.train()
        perm = torch.randperm(n, device=device); total = 0.0
        for i in range(0, n, batch_run):
            idx = perm[i:i + batch_run]
            if len(idx) < 2:
                continue
            xb = augment(prep(Xtr_d[idx])); yb = ytr_d[idx]
            with autocast(device, use_amp):
                loss = loss_fn(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            total += loss.item() * len(idx)
        sched.step()
        s = summarize(y[va], predict_logits(model, X[va], device, use_amp).argmax(1), class_names)
        print(f"  fine-tune epoch {epoch + 1}/{ft_epochs_run}  train_loss {total / n:.3f}  "
              f"val_bal_acc {s['balanced_acc']:.3f}  val_acc {s['accuracy']:.3f}  ({time.time() - t0:.0f}s)")
        if s["balanced_acc"] > best_score:
            best_score, best_epoch = s["balanced_acc"], epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    for m, name in prunable(model):            # make the zeros permanent
        try:
            prune.remove(m, name)
        except ValueError:
            pass
    tot = zer = 0
    for m, _ in prunable(model):
        tot += m.weight.numel(); zer += int((m.weight == 0).sum().item())
    print(f"  best fine-tune epoch = {best_epoch};  weights that are exactly zero: {100 * zer / tot:.1f}% of {tot:,}")
    return model, zer / tot


for seed in seeds_to_run:
    out_path = os.path.join(PRED_DIR, f"{PRUNE_NAME}_seed{seed}.npz")
    ck_path = os.path.join(CKPT_DIR, f"{PRUNE_NAME}_seed{seed}.pt")
    if os.path.exists(out_path) and os.path.exists(ck_path) and not SMOKE:
        print(f"seed {seed}: pruned model already done, skipping"); continue
    base_path = os.path.join(CKPT_DIR, f"baseline_seed{seed}.pt")
    assert os.path.exists(base_path), f"{base_path} not found. Train the baseline first (Cell 5)."
    base_state = torch.load(base_path, map_location="cpu")
    base_model = build_resnet50(len(class_names)); base_model.load_state_dict(base_state)
    base_nz, base_mb = count_nonzero_params(base_model), state_size_mb(base_model)
    print(f"\n===== seed {seed}: baseline has {base_nz:,} non-zero parameters, file {base_mb:.0f} MB =====")

    t0 = time.time()
    pmodel, sparsity = prune_and_finetune(base_state, seed)
    use_amp = device.type == "cuda"
    va_l = predict_logits(pmodel, X[va], device, use_amp)
    te_l = predict_logits(pmodel, X[te], device, use_amp)
    nz = count_nonzero_params(pmodel)
    np.savez_compressed(out_path, val_logits=va_l, test_logits=te_l, y_val=y[va], y_test=y[te],
                        lesion_test=lesion[te], sparsity=sparsity, nonzero_params=nz,
                        size_mb_dense=state_size_mb(pmodel))
    torch.save({k: v.cpu() for k, v in pmodel.state_dict().items()}, ck_path)     # NEW: keep the pruned model
    st = summarize(y[te], te_l.argmax(1), class_names)
    print(f"  non-zero parameters: {nz:,} ({base_nz / nz:.1f}x fewer than baseline). "
          f"The saved file stays {state_size_mb(pmodel):.0f} MB (dense storage), so the FILE is not smaller.")
    print(f"  TEST (sanity only): acc {st['accuracy']:.3f} bal_acc {st['balanced_acc']:.3f} "
          f"tail_recall {st['tail_recall']:.3f} mel_recall {st['mel_recall']:.3f}")
    print(f"  took {(time.time() - t0) / 60:.1f} min")
    del pmodel
    if device.type == "cuda":
        torch.cuda.empty_cache()
print("\nDONE (pruning).")

In [ ]:
#cell 8
N_CALIB = 64 if SMOKE else 500

def quantize_int8(base_state, calib_X):
    from torchvision.models.quantization import resnet50 as q_resnet50
    torch.backends.quantized.engine = "fbgemm"
    m = q_resnet50(weights=None, quantize=False)
    m.fc = nn.Linear(m.fc.in_features, len(class_names))
    m.load_state_dict(base_state)
    m.eval()
    m.fuse_model()                                              # merge conv + batchnorm + relu
    m.qconfig = torch.ao.quantization.get_default_qconfig("fbgemm")
    torch.ao.quantization.prepare(m, inplace=True)
    with torch.no_grad():                                       # calibration: run TRAIN images through
        for i in range(0, len(calib_X), 64):
            m(prep(torch.from_numpy(calib_X[i:i + 64])))
    torch.ao.quantization.convert(m, inplace=True)
    return m

cpu = torch.device("cpu")
for seed in seeds_to_run:
    out_path = os.path.join(PRED_DIR, f"int8_seed{seed}.npz")
    if os.path.exists(out_path) and not SMOKE:
        print(f"seed {seed}: INT8 already done, skipping"); continue
    base_path = os.path.join(CKPT_DIR, f"baseline_seed{seed}.pt")
    assert os.path.exists(base_path), f"{base_path} not found. Train the baseline first (Cell 5)."
    base_state = torch.load(base_path, map_location="cpu")
    base_model = build_resnet50(len(class_names)); base_model.load_state_dict(base_state)
    base_mb = state_size_mb(base_model)

    t0 = time.time()
    rng = np.random.default_rng(seed)
    calib_idx = rng.choice(tr, size=min(N_CALIB, len(tr)), replace=False)    # TRAIN images only
    try:
        qmodel = quantize_int8(base_state, X[calib_idx])
    except Exception as e:
        print("INT8 quantization failed. Please report this error message:\n", repr(e))
        raise
    va_l = predict_logits(qmodel, X[va], cpu, False)
    te_l = predict_logits(qmodel, X[te], cpu, False)
    qmb = state_size_mb(qmodel)
    np.savez_compressed(out_path, val_logits=va_l, test_logits=te_l, y_val=y[va], y_test=y[te],
                        lesion_test=lesion[te], size_mb=qmb)
    st = summarize(y[te], te_l.argmax(1), class_names)
    print(f"seed {seed}: baseline file {base_mb:.0f} MB -> INT8 file {qmb:.0f} MB ({base_mb / qmb:.1f}x smaller)")
    print(f"  TEST (sanity only): acc {st['accuracy']:.3f} bal_acc {st['balanced_acc']:.3f} "
          f"tail_recall {st['tail_recall']:.3f} mel_recall {st['mel_recall']:.3f}")
    print(f"  took {(time.time() - t0) / 60:.1f} min")
print("\nDONE (INT8).")

In [ ]:
#cell 9
STUDENT_CANDIDATES = ["resnet18", "mobilenet_v3_large", "efficientnet_b0", "efficientnet_b1",
                      "regnet_y_800mf", "regnet_y_1_6gf"]
TEMPERATURE = 4.0
ALPHA = 0.5            # weight of the distillation (KL) part; (1 - ALPHA) goes to normal cross-entropy
n_classes = len(class_names)

def replace_head(m, n):
    if hasattr(m, "fc") and isinstance(m.fc, nn.Linear):
        m.fc = nn.Linear(m.fc.in_features, n)
    elif hasattr(m, "classifier") and isinstance(m.classifier, nn.Sequential) and isinstance(m.classifier[-1], nn.Linear):
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n)
    else:
        raise RuntimeError("unknown classifier-head layout for this model")
    return m

def make_student(name, n, pretrained):
    ctor = getattr(torchvision.models, name)
    if pretrained:
        try:
            m = ctor(weights="DEFAULT")                     # ImageNet weights
        except Exception as e:
            raise RuntimeError("Could not download pre-trained weights for " + name + ": " + str(e))
    else:
        m = ctor(weights=None)
    return replace_head(m, n)

# ---- RULE (fixed in advance): pick the candidate whose FP32 file size is closest to (teacher size / 4) ----
teacher_mb = state_size_mb(build_resnet50(n_classes))
target_mb = teacher_mb / 4
rows = []
for name in STUDENT_CANDIDATES:
    m = make_student(name, n_classes, pretrained=False)
    rows.append((name, sum(p.numel() for p in m.parameters()), state_size_mb(m)))
print(f"teacher ResNet-50: {teacher_mb:.1f} MB | target student size = teacher/4 = {target_mb:.1f} MB")
for name, npar, mb in rows:
    print(f"  {name:20s} {npar / 1e6:5.2f} M parameters  {mb:6.1f} MB  ({teacher_mb / mb:.1f}x smaller than the teacher)")
STUDENT_NAME = min(rows, key=lambda r: abs(r[2] - target_mb))[0]
print("\n>>> chosen student (closest to the target size):", STUDENT_NAME)

def distill_one_seed(seed):
    use_amp = device.type == "cuda"
    set_seed(seed)
    teacher = build_resnet50(n_classes)
    teacher.load_state_dict(torch.load(os.path.join(CKPT_DIR, f"baseline_seed{seed}.pt"), map_location="cpu"))
    teacher.to(device).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)

    student = make_student(STUDENT_NAME, n_classes, pretrained=True).to(device)
    opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs_run)
    scaler = make_scaler(use_amp)
    ce = nn.CrossEntropyLoss()                         # plain cross-entropy, no class weights
    Xtr_d = torch.from_numpy(X[tr]).to(device); ytr_d = torch.from_numpy(y[tr]).to(device)
    n = len(Xtr_d)
    T = TEMPERATURE

    best_score, best_epoch, best_state = -1.0, -1, None
    for epoch in range(epochs_run):
        t0 = time.time(); student.train()
        perm = torch.randperm(n, device=device); total = 0.0
        for i in range(0, n, batch_run):
            idx = perm[i:i + batch_run]
            if len(idx) < 2:
                continue
            xb = augment(prep(Xtr_d[idx])); yb = ytr_d[idx]
            with torch.no_grad():
                with autocast(device, use_amp):
                    t_logits = teacher(xb)
            with autocast(device, use_amp):
                s_logits = student(xb)
            s32, t32 = s_logits.float(), t_logits.float()
            kl = F.kl_div(F.log_softmax(s32 / T, dim=1), F.softmax(t32 / T, dim=1),
                          reduction="batchmean") * (T * T)       # standard T^2 scaling
            loss = ALPHA * kl + (1 - ALPHA) * ce(s32, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            total += loss.item() * len(idx)
        sched.step()
        s = summarize(y[va], predict_logits(student, X[va], device, use_amp).argmax(1), class_names)
        print(f"seed {seed} epoch {epoch + 1:2d}/{epochs_run}  loss {total / n:.3f}  "
              f"val_acc {s['accuracy']:.3f}  val_bal_acc {s['balanced_acc']:.3f}  ({time.time() - t0:.0f}s)")
        if s["balanced_acc"] > best_score:               # choose the epoch on VALIDATION only
            best_score, best_epoch = s["balanced_acc"], epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}

    student.load_state_dict(best_state)
    print(f"seed {seed}: best epoch = {best_epoch}  (val balanced accuracy {best_score:.3f})")
    torch.save(best_state, os.path.join(CKPT_DIR, f"distilled_seed{seed}.pt"))
    va_l = predict_logits(student, X[va], device, use_amp)
    te_l = predict_logits(student, X[te], device, use_amp)
    nparams = int(sum(p.numel() for p in student.parameters()))
    np.savez_compressed(os.path.join(PRED_DIR, f"distilled_seed{seed}.npz"),
                        val_logits=va_l, test_logits=te_l, y_val=y[va], y_test=y[te],
                        lesion_test=lesion[te], size_mb=state_size_mb(student),
                        nonzero_params=count_nonzero_params(student), student_name=STUDENT_NAME)
    st = summarize(y[te], te_l.argmax(1), class_names)
    print(f"seed {seed} TEST (sanity only): acc {st['accuracy']:.3f}  bal_acc {st['balanced_acc']:.3f}  "
          f"tail_recall {st['tail_recall']:.3f}  mel_recall {st['mel_recall']:.3f}")
    print(f"  student file {state_size_mb(student):.1f} MB, {nparams / 1e6:.2f} M parameters "
          f"({teacher_mb / state_size_mb(student):.1f}x smaller than the teacher)")

for seed in seeds_to_run:
    done = (os.path.exists(os.path.join(CKPT_DIR, f"distilled_seed{seed}.pt")) and
            os.path.exists(os.path.join(PRED_DIR, f"distilled_seed{seed}.npz")))
    if done and not SMOKE:
        print(f"seed {seed}: distilled model already done, skipping"); continue
    assert os.path.exists(os.path.join(CKPT_DIR, f"baseline_seed{seed}.pt")), "Train the baseline first (Cell 5)."
    t0 = time.time()
    distill_one_seed(seed)
    print(f"seed {seed} finished in {(time.time() - t0) / 60:.1f} min\n")
print("DONE (distillation).")

In [ ]:
#cell 10
import itertools
import pandas as pd

MODELS = ["baseline", "pruned75", "int8", "distilled"]
N_BOOT = 100 if SMOKE else 1000
RESULT_DIR = os.path.join(RUN_DIR, "results"); os.makedirs(RESULT_DIR, exist_ok=True)
METRICS = ["tail_recall", "mel_recall", "balanced_acc", "accuracy", "macro_f1"]
cidx = {c: i for i, c in enumerate(class_names)}
TAIL_IDX = [cidx[c] for c in TAIL_CLASSES]
MEL_IDX = cidx[MEL_CLASS]


def lesion_bootstrap_weights(lesion_ids, n_boot, seed=0):
    """(n_boot, n_images) weights: draw LESIONS with replacement; every image of a drawn lesion comes along."""
    uniq, inv = np.unique(lesion_ids, return_inverse=True)
    L = len(uniq)
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, L, size=(n_boot, L))
    counts = np.zeros((n_boot, L))
    np.add.at(counts, (np.repeat(np.arange(n_boot), L), draws.ravel()), 1.0)
    return counts[:, inv]


def weighted_metrics(pred, y_true, W, n_classes, tail_idx, mel_idx):
    """All metrics for every row of the weight matrix W (n_rows, n_images). Returns dict of arrays."""
    correct = (pred == y_true).astype(float)
    rec = np.full((W.shape[0], n_classes), np.nan)
    prec = np.zeros((W.shape[0], n_classes))
    for c in range(n_classes):
        is_c = (y_true == c).astype(float)
        den = W @ is_c
        rec[:, c] = np.where(den > 0, (W @ (correct * is_c)) / np.maximum(den, 1e-12), np.nan)
        pc = (pred == c).astype(float)
        pden = W @ pc
        prec[:, c] = np.where(pden > 0, (W @ (correct * pc)) / np.maximum(pden, 1e-12), 0.0)
    with np.errstate(invalid="ignore", divide="ignore"):
        f1 = np.where((prec + rec) > 0, 2 * prec * rec / (prec + rec), 0.0)
    f1 = np.where(np.isnan(rec), np.nan, f1)
    tm = np.isin(y_true, tail_idx).astype(float)
    mm = (y_true == mel_idx).astype(float)
    out = {
        "accuracy": (W @ correct) / W.sum(1),
        "balanced_acc": np.nanmean(rec, axis=1),
        "macro_f1": np.nanmean(f1, axis=1),
        "tail_recall": (W @ (correct * tm)) / np.maximum(W @ tm, 1e-12),
        "mel_recall": (W @ (correct * mm)) / np.maximum(W @ mm, 1e-12),
        "rec": rec,
    }
    return out


# ----------------------------- self-test (must pass before we trust any number) -----------------------------
def _selftest():
    rng = np.random.default_rng(1)
    n, nc = 400, 7
    yt = rng.integers(0, nc, n)
    les = rng.integers(0, 150, n)
    Wb = lesion_bootstrap_weights(les, 50, seed=3)
    W1 = np.ones((1, n))
    m = weighted_metrics(yt.copy(), yt, W1, nc, [0, 3, 6], 4)           # perfect predictor
    assert abs(m["accuracy"][0] - 1) < 1e-9 and abs(m["tail_recall"][0] - 1) < 1e-9 and abs(m["macro_f1"][0] - 1) < 1e-9
    m = weighted_metrics(np.full(n, 5), yt, W1, nc, [0, 3, 6], 4)       # "always class 5"
    assert m["tail_recall"][0] == 0 and m["mel_recall"][0] == 0
    assert abs(m["accuracy"][0] - (yt == 5).mean()) < 1e-9
    mb = weighted_metrics(yt.copy(), yt, Wb, nc, [0, 3, 6], 4)          # perfect predictor in every resample
    assert np.allclose(mb["accuracy"], 1) and np.allclose(mb["tail_recall"], 1)
    print("self-test of the metric code: PASSED")

_selftest()

# ----------------------------- load what exists -----------------------------
data = {}                                   # data[model][seed] = (pred, y_true, lesion, extra)
for mdl in MODELS:
    for pth in sorted(glob.glob(os.path.join(PRED_DIR, f"{mdl}_seed*.npz"))):
        sd = int(os.path.basename(pth).split("seed")[1].split(".")[0])
        z = np.load(pth, allow_pickle=True)
        data.setdefault(mdl, {})[sd] = (z["test_logits"].argmax(1), z["y_test"], z["lesion_test"], z)
models_found = [m for m in MODELS if m in data]
print("models found:", {m: sorted(data[m]) for m in models_found})
assert "baseline" in data, "no baseline predictions found (run Cell 5 first)"

ref_seed = sorted(data["baseline"])[0]
y_ref, les_ref = data["baseline"][ref_seed][1], data["baseline"][ref_seed][2]
for m in models_found:
    for sd, (pr, yy, ll, _) in data[m].items():
        assert (yy == y_ref).all() and (ll == les_ref).all(), "test sets differ between models/seeds!"

W_boot = lesion_bootstrap_weights(les_ref, N_BOOT, seed=0)             # SAME resamples for all models (paired)
W_one = np.ones((1, len(y_ref)))

point, boot = {}, {}                        # point[m][metric] = mean over seeds; boot[m][metric] = (N_BOOT,)
seed_sd, per_class_point, per_class_boot = {}, {}, {}
for m in models_found:
    pts, bts, recs_p, recs_b = [], [], [], []
    for sd in sorted(data[m]):
        pr = data[m][sd][0]
        p = weighted_metrics(pr, y_ref, W_one, len(class_names), TAIL_IDX, MEL_IDX)
        b = weighted_metrics(pr, y_ref, W_boot, len(class_names), TAIL_IDX, MEL_IDX)
        pts.append({k: p[k][0] for k in METRICS}); bts.append({k: b[k] for k in METRICS})
        recs_p.append(p["rec"][0]); recs_b.append(b["rec"])
    point[m] = {k: float(np.mean([d[k] for d in pts])) for k in METRICS}
    seed_sd[m] = {k: float(np.std([d[k] for d in pts])) for k in METRICS}
    boot[m] = {k: np.mean([d[k] for d in bts], axis=0) for k in METRICS}
    per_class_point[m] = np.mean(recs_p, axis=0)
    per_class_boot[m] = np.nanmean(np.stack(recs_b), axis=0)

def ci(a):
    return float(np.nanpercentile(a, 2.5)), float(np.nanpercentile(a, 97.5))

# ----------------------------- table 1: main metrics -----------------------------
rows = []
for m in models_found:
    for k in METRICS:
        lo, hi = ci(boot[m][k])
        rows.append(dict(model=m, metric=k, mean=point[m][k], ci_low=lo, ci_high=hi,
                         seed_sd=seed_sd[m][k], n_seeds=len(data[m])))
main_df = pd.DataFrame(rows)
main_df.to_csv(os.path.join(RESULT_DIR, "main_metrics.csv"), index=False)
print(f"\n=== MAIN METRICS (mean over {len(data['baseline'])} seed(s); 95% CI from {N_BOOT} lesion-level bootstrap repeats) ===")
for k in METRICS:
    print(f"\n{k}:")
    for m in models_found:
        r = main_df[(main_df.model == m) & (main_df.metric == k)].iloc[0]
        print(f"  {m:10s} {r['mean']:.3f}  [{r['ci_low']:.3f}, {r['ci_high']:.3f}]   (sd over seeds {r['seed_sd']:.3f})")

# ----------------------------- table 2: per-class recall -----------------------------
rows = []
for m in models_found:
    for c, name in enumerate(class_names):
        lo, hi = ci(per_class_boot[m][:, c])
        rows.append(dict(model=m, cls=name, recall=float(per_class_point[m][c]), ci_low=lo, ci_high=hi))
pc_df = pd.DataFrame(rows)
pc_df.to_csv(os.path.join(RESULT_DIR, "per_class_recall.csv"), index=False)
print("\n=== PER-CLASS RECALL (test) ===  (df has only 12 and vasc 13 test images: very wide intervals)")
piv = pc_df.assign(txt=lambda d: d.apply(lambda r: f"{r.recall:.2f} [{r.ci_low:.2f},{r.ci_high:.2f}]", axis=1)) \
           .pivot(index="cls", columns="model", values="txt")[models_found]
print(piv.to_string())

# ----------------------------- table 3: paired differences -----------------------------
pairs = [(a, "baseline") for a in models_found if a != "baseline"]
pairs += [(a, b) for a, b in itertools.combinations([m for m in models_found if m != "baseline"], 2)]
rows = []
for a, b in pairs:
    for k in ["tail_recall", "mel_recall", "balanced_acc", "accuracy"]:
        diff_b = boot[a][k] - boot[b][k]
        lo, hi = ci(diff_b)
        rows.append(dict(comparison=f"{a} - {b}", metric=k, diff=point[a][k] - point[b][k],
                         ci_low=lo, ci_high=hi, excludes_zero=bool(lo > 0 or hi < 0)))
pair_df = pd.DataFrame(rows)
pair_df.to_csv(os.path.join(RESULT_DIR, "paired_differences.csv"), index=False)
print("\n=== PAIRED DIFFERENCES (same resampled lesions for both models) ===")
print("Only say 'clearly different' when the interval excludes zero (excludes_zero = True).\n")
print(pair_df.round(3).to_string(index=False))

# ----------------------------- table 4: sizes -----------------------------
rows = []
for m in models_found:
    sd0 = sorted(data[m])[0]
    z = data[m][sd0][3]
    if m == "baseline":
        mdl = build_resnet50(len(class_names)); mdl.load_state_dict(torch.load(os.path.join(CKPT_DIR, f"baseline_seed{sd0}.pt"), map_location="cpu"))
        size, nz = state_size_mb(mdl), count_nonzero_params(mdl)
    elif m.startswith("pruned"):
        size, nz = float(z["size_mb_dense"]), int(z["nonzero_params"])
    elif m == "int8":
        size, nz = float(z["size_mb"]), np.nan
    else:
        size, nz = float(z["size_mb"]), int(z["nonzero_params"])
    rows.append(dict(model=m, file_size_mb=round(size, 1), nonzero_params=nz))
size_df = pd.DataFrame(rows)
size_df.to_csv(os.path.join(RESULT_DIR, "sizes.csv"), index=False)
print("\n=== MODEL SIZES ===  (pruning: the DENSE file does not shrink; judge it by non-zero parameters)")
print(size_df.to_string(index=False))
print("\nSaved CSV files in", RESULT_DIR)

In [ ]:
#cell for validation

import os, glob, numpy as np
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/RareClassRecall_SkinLesion'
PRED_DIR = os.path.join(PROJECT_DIR, 'run', 'preds')
print("prediction files found:", len(glob.glob(os.path.join(PRED_DIR, '*.npz'))))

class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']      # same alphabetical order as before
TAIL = [class_names.index(c) for c in ['akiec', 'df', 'vasc']]
MEL = class_names.index('mel')

def val_metrics(y, pred):
    acc = (y == pred).mean()
    rec = np.array([(pred[y == c] == c).mean() if (y == c).any() else np.nan for c in range(len(class_names))])
    tm = np.isin(y, TAIL); mm = (y == MEL)
    return acc, np.nanmean(rec), (pred[tm] == y[tm]).mean(), (pred[mm] == y[mm]).mean()

print(f"\n{'model':10s} {'seed':>4s}  {'val_acc':>7s} {'val_bal':>7s} {'val_tail':>8s} {'val_mel':>7s}  agree_with_baseline")
for mdl in ["baseline", "pruned75", "int8", "distilled"]:
    for pth in sorted(glob.glob(os.path.join(PRED_DIR, f"{mdl}_seed*.npz"))):
        sd = int(os.path.basename(pth).split("seed")[1].split(".")[0])
        z = np.load(pth, allow_pickle=True)
        pred = z["val_logits"].argmax(1)
        acc, bal, tail, mel = val_metrics(z["y_val"], pred)
        bpath = os.path.join(PRED_DIR, f"baseline_seed{sd}.npz")
        agree = (pred == np.load(bpath)["val_logits"].argmax(1)).mean() if os.path.exists(bpath) else float("nan")
        print(f"{mdl:10s} {sd:4d}  {acc:7.3f} {bal:7.3f} {tail:8.3f} {mel:7.3f}  {agree:.3f}")
print("\n'agree_with_baseline' = share of validation images where the compressed model gives the same answer as the baseline.")

In [ ]:
# cell 11

from PIL import Image, ImageFilter

N_CALIB = 64 if SMOKE else 500
CORRUPTIONS = [("blur", 1.0), ("blur", 2.0), ("blur", 3.0),
               ("brightness", 0.5), ("brightness", 1.5),
               ("noise", 10.0), ("noise", 20.0), ("noise", 40.0)]      # levels fixed in advance
cond_key = lambda kind, lvl: f"{kind}_{lvl}"

def corrupt(imgs, kind, level, seed=123):
    """imgs: uint8 (N,H,W,3). The SAME corrupted images are used for every model and seed."""
    if kind == "blur":                          # Gaussian blur, radius = sigma in pixels
        out = np.empty_like(imgs)
        for i in range(len(imgs)):
            out[i] = np.asarray(Image.fromarray(imgs[i]).filter(ImageFilter.GaussianBlur(radius=level)))
        return out
    if kind == "brightness":                    # multiply pixel values, clip to 0-255
        return np.clip(imgs.astype(np.float32) * level, 0, 255).astype(np.uint8)
    if kind == "noise":                         # add Gaussian noise (sigma in 0-255 units), clip
        rng = np.random.default_rng(seed)
        return np.clip(imgs.astype(np.float32) + rng.normal(0, level, imgs.shape).astype(np.float32),
                       0, 255).astype(np.uint8)
    raise ValueError(kind)

# ---- build the corrupted test sets once ----
X_test = X[te]
corr_sets = {"clean": X_test}
for kind, lvl in CORRUPTIONS:
    corr_sets[cond_key(kind, lvl)] = corrupt(X_test, kind, lvl)
print("conditions:", list(corr_sets))
print("mean pixel value per condition:", {k: round(float(v.mean()), 1) for k, v in corr_sets.items()})

# ---- model builders (self-contained, so this cell works after a restart of Cells 2 + 4) ----
def replace_head(m, n):
    if hasattr(m, "fc") and isinstance(m.fc, nn.Linear):
        m.fc = nn.Linear(m.fc.in_features, n)
    elif hasattr(m, "classifier") and isinstance(m.classifier, nn.Sequential) and isinstance(m.classifier[-1], nn.Linear):
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n)
    else:
        raise RuntimeError("unknown head layout")
    return m

def make_student_nopre(name, n):
    return replace_head(getattr(torchvision.models, name)(weights=None), n)

def quantize_int8(base_state, calib_X):
    from torchvision.models.quantization import resnet50 as q_resnet50
    torch.backends.quantized.engine = "fbgemm"
    m = q_resnet50(weights=None, quantize=False)
    m.fc = nn.Linear(m.fc.in_features, len(class_names))
    m.load_state_dict(base_state)
    m.eval()
    m.fuse_model()
    m.qconfig = torch.ao.quantization.get_default_qconfig("fbgemm")
    torch.ao.quantization.prepare(m, inplace=True)
    with torch.no_grad():
        for i in range(0, len(calib_X), 64):
            m(prep(torch.from_numpy(calib_X[i:i + 64])))
    torch.ao.quantization.convert(m, inplace=True)
    return m

cpu = torch.device("cpu")
use_amp = device.type == "cuda"
ROB_DIR = os.path.join(RUN_DIR, "robust"); os.makedirs(ROB_DIR, exist_ok=True)

def eval_all_conditions(model, dev, amp):
    return {k: predict_logits(model, v, dev, amp) for k, v in corr_sets.items()}

seeds_avail = sorted(int(f.split("seed")[1].split(".")[0]) for f in os.listdir(CKPT_DIR) if f.startswith("baseline_seed"))
print("seeds with a trained baseline:", seeds_avail)
for seed in seeds_avail:
    base_state = torch.load(os.path.join(CKPT_DIR, f"baseline_seed{seed}.pt"), map_location="cpu")
    for mdl in ["baseline", "pruned75", "int8", "distilled"]:
        out_path = os.path.join(ROB_DIR, f"robust_{mdl}_seed{seed}.npz")
        if os.path.exists(out_path) and not SMOKE:
            print(f"{mdl} seed {seed}: already done, skipping"); continue
        t0 = time.time()
        if mdl == "baseline":
            m = build_resnet50(len(class_names)); m.load_state_dict(base_state); m.to(device)
            dev, amp = device, use_amp
        elif mdl == "pruned75":
            ck = os.path.join(CKPT_DIR, f"pruned75_seed{seed}.pt")
            if not os.path.exists(ck):
                print(f"pruned75 seed {seed}: checkpoint missing (re-run the patched Cell 7), skipping"); continue
            m = build_resnet50(len(class_names)); m.load_state_dict(torch.load(ck, map_location="cpu")); m.to(device)
            dev, amp = device, use_amp
        elif mdl == "int8":
            rng = np.random.default_rng(seed)                          # same calibration images as Cell 8
            calib_idx = rng.choice(tr, size=min(N_CALIB, len(tr)), replace=False)
            m = quantize_int8(base_state, X[calib_idx])
            dev, amp = cpu, False
        else:
            ck = os.path.join(CKPT_DIR, f"distilled_seed{seed}.pt")
            pp = os.path.join(PRED_DIR, f"distilled_seed{seed}.npz")
            if not (os.path.exists(ck) and os.path.exists(pp)):
                print(f"distilled seed {seed}: files missing (run Cell 9), skipping"); continue
            sname = str(np.load(pp, allow_pickle=True)["student_name"])
            m = make_student_nopre(sname, len(class_names)); m.load_state_dict(torch.load(ck, map_location="cpu")); m.to(device)
            dev, amp = device, use_amp
        logits = eval_all_conditions(m, dev, amp)

        # sanity: the clean logits recomputed here must agree with the ones saved earlier
        saved = np.load(os.path.join(PRED_DIR, f"{mdl}_seed{seed}.npz"), allow_pickle=True)["test_logits"]
        agree = float((logits["clean"].argmax(1) == saved.argmax(1)).mean())
        print(f"{mdl:10s} seed {seed}: clean predictions agree with the saved ones: {agree:.4f}  "
              f"({time.time() - t0:.0f}s for {len(corr_sets)} conditions)")
        np.savez_compressed(out_path, y_test=y[te], lesion_test=lesion[te], **logits)
        del m
        if device.type == "cuda":
            torch.cuda.empty_cache()
print("\nDONE (robustness logits saved in", ROB_DIR, ")")

In [ ]:
#cell 12
import os, glob, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/RareClassRecall_SkinLesion'
RUN_DIR = os.path.join(PROJECT_DIR, 'run')
RES_DIR = os.path.join(RUN_DIR, 'results')          # made by Cell 10
ROB_DIR = os.path.join(RUN_DIR, 'robust')           # made by Cell 11
FIG_DIR = os.path.join(RUN_DIR, 'figures'); os.makedirs(FIG_DIR, exist_ok=True)

class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
TAIL_IDX = [class_names.index(c) for c in ['akiec', 'df', 'vasc']]
MEL_IDX = class_names.index('mel')
MODELS = ["baseline", "pruned75", "int8", "distilled"]
LABEL = {"baseline": "Baseline ResNet-50", "pruned75": "Pruned 75%", "int8": "INT8 (post-training)", "distilled": "Distilled student"}
COLOR = {"baseline": "#444444", "pruned75": "#1f77b4", "int8": "#d62728", "distilled": "#2ca02c"}
MARK = {"baseline": "o", "pruned75": "s", "int8": "^", "distilled": "D"}
N_BOOT = 1000
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "legend.fontsize": 7})


# ---------- same metric code as Cell 10 (self-contained) ----------
def lesion_bootstrap_weights(lesion_ids, n_boot, seed=0):
    uniq, inv = np.unique(lesion_ids, return_inverse=True)
    L = len(uniq)
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, L, size=(n_boot, L))
    counts = np.zeros((n_boot, L))
    np.add.at(counts, (np.repeat(np.arange(n_boot), L), draws.ravel()), 1.0)
    return counts[:, inv]

def weighted_metrics(pred, y_true, W, n_classes, tail_idx, mel_idx):
    correct = (pred == y_true).astype(float)
    rec = np.full((W.shape[0], n_classes), np.nan)
    for c in range(n_classes):
        is_c = (y_true == c).astype(float)
        den = W @ is_c
        rec[:, c] = np.where(den > 0, (W @ (correct * is_c)) / np.maximum(den, 1e-12), np.nan)
    tm = np.isin(y_true, tail_idx).astype(float)
    mm = (y_true == mel_idx).astype(float)
    return {"accuracy": (W @ correct) / W.sum(1), "balanced_acc": np.nanmean(rec, axis=1),
            "tail_recall": (W @ (correct * tm)) / np.maximum(W @ tm, 1e-12),
            "mel_recall": (W @ (correct * mm)) / np.maximum(W @ mm, 1e-12)}

def ci(a):
    return float(np.nanpercentile(a, 2.5)), float(np.nanpercentile(a, 97.5))


# =============================== PART A: main table ===============================
main = pd.read_csv(os.path.join(RES_DIR, "main_metrics.csv"))
sizes = pd.read_csv(os.path.join(RES_DIR, "sizes.csv"))
models_main = [m for m in MODELS if m in set(main.model)]
n_seeds = int(main.n_seeds.max())

def txt(m, k):
    r = main[(main.model == m) & (main.metric == k)].iloc[0]
    return f"{r['mean']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]"

rows = []
for m in models_main:
    s = sizes[sizes.model == m].iloc[0]
    nz = "n/a" if pd.isna(s.nonzero_params) else f"{s.nonzero_params / 1e6:.2f}"
    rows.append({"Model": LABEL[m], "File (MB)": f"{s.file_size_mb:.1f}", "Non-zero params (M)": nz,
                 "Tail recall": txt(m, "tail_recall"), "Melanoma recall": txt(m, "mel_recall"),
                 "Balanced acc.": txt(m, "balanced_acc"), "Accuracy": txt(m, "accuracy")})
table = pd.DataFrame(rows)
table.to_csv(os.path.join(FIG_DIR, "table_main.csv"), index=False)
print(f"=== MAIN TABLE (mean over {n_seeds} seeds; 95% CI from lesion-level bootstrap) ===")
print(table.to_string(index=False))
tex = ["\\begin{tabular}{lrrcccc}", "\\hline", " & ".join(table.columns).replace("%", "\\%") + " \\\\", "\\hline"]
for _, r in table.iterrows():
    tex.append(" & ".join(str(v) for v in r.values) + " \\\\")
tex += ["\\hline", "\\end{tabular}"]
open(os.path.join(FIG_DIR, "table_main.tex"), "w").write("\n".join(tex))
print("\nSaved table_main.csv and table_main.tex in", FIG_DIR)


# =============================== PART B: figure 1 (main results) ===============================
fig, ax = plt.subplots(figsize=(6.5, 2.8))
metrics_show = [("tail_recall", "Tail-group recall"), ("mel_recall", "Melanoma recall"), ("balanced_acc", "Balanced accuracy")]
width = 0.8 / len(models_main)
for j, m in enumerate(models_main):
    xs, means, lo, hi = [], [], [], []
    for i, (k, _) in enumerate(metrics_show):
        r = main[(main.model == m) & (main.metric == k)].iloc[0]
        xs.append(i + (j - (len(models_main) - 1) / 2) * width)
        means.append(r["mean"]); lo.append(r["mean"] - r["ci_low"]); hi.append(r["ci_high"] - r["mean"])
    ax.bar(xs, means, width * 0.95, yerr=[lo, hi], capsize=2, color=COLOR[m], label=LABEL[m], alpha=0.9,
           error_kw={"elinewidth": 0.8})
ax.set_xticks(range(len(metrics_show))); ax.set_xticklabels([n for _, n in metrics_show])
ax.set_ylabel("Score (test set)"); ax.set_ylim(0, 1)
ax.legend(ncol=2, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.22))
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig1_main_results.png"), dpi=300, bbox_inches="tight")
fig.savefig(os.path.join(FIG_DIR, "fig1_main_results.pdf"), bbox_inches="tight")
plt.show()


# =============================== PART C: robustness ===============================
files = {m: sorted(glob.glob(os.path.join(ROB_DIR, f"robust_{m}_seed*.npz"))) for m in MODELS}
models_r = [m for m in MODELS if files[m]]
if not models_r:
    print("No robustness files found (run Cell 11 first). Skipping the robustness part.")
else:
    data_r, y_ref, les_ref, conds = {}, None, None, None
    for m in models_r:
        data_r[m] = []
        for f in files[m]:
            z = np.load(f, allow_pickle=True)
            if y_ref is None:
                y_ref, les_ref = z["y_test"], z["lesion_test"]
                conds = [k for k in z.files if k not in ("y_test", "lesion_test")]
            assert (z["y_test"] == y_ref).all() and (z["lesion_test"] == les_ref).all(), "test sets differ"
            data_r[m].append({c: z[c].argmax(1) for c in conds})
    print("\nrobustness: models", models_r, "| seeds per model", {m: len(data_r[m]) for m in models_r}, "| conditions", conds)

    RM = ["tail_recall", "mel_recall", "balanced_acc", "accuracy"]
    W_boot = lesion_bootstrap_weights(les_ref, N_BOOT, seed=0)
    W_one = np.ones((1, len(y_ref)))
    pt, bt = {}, {}
    for m in models_r:
        for c in conds:
            ps = [weighted_metrics(d[c], y_ref, W_one, 7, TAIL_IDX, MEL_IDX) for d in data_r[m]]
            bs = [weighted_metrics(d[c], y_ref, W_boot, 7, TAIL_IDX, MEL_IDX) for d in data_r[m]]
            pt[(m, c)] = {k: float(np.mean([p[k][0] for p in ps])) for k in RM}
            bt[(m, c)] = {k: np.mean([b[k] for b in bs], axis=0) for k in RM}

    rows = []
    for m in models_r:
        for c in conds:
            for k in RM:
                lo, hi = ci(bt[(m, c)][k])
                row = dict(model=m, condition=c, metric=k, mean=pt[(m, c)][k], ci_low=lo, ci_high=hi)
                if c != "clean":
                    dlo, dhi = ci(bt[(m, c)][k] - bt[(m, "clean")][k])
                    row.update(drop_vs_clean=pt[(m, c)][k] - pt[(m, "clean")][k], drop_ci_low=dlo, drop_ci_high=dhi)
                rows.append(row)
    rob_df = pd.DataFrame(rows)
    rob_df.to_csv(os.path.join(FIG_DIR, "robustness_metrics.csv"), index=False)

    for k in ["tail_recall", "balanced_acc"]:
        print(f"\n--- {k}: value (change from clean [95% CI]) ---")
        tab = {}
        for m in models_r:
            tab[LABEL[m]] = {}
            for c in conds:
                r = rob_df[(rob_df.model == m) & (rob_df.condition == c) & (rob_df.metric == k)].iloc[0]
                tab[LABEL[m]][c] = f"{r['mean']:.3f}" if c == "clean" else \
                    f"{r['mean']:.3f} ({r['drop_vs_clean']:+.3f} [{r['drop_ci_low']:+.3f},{r['drop_ci_high']:+.3f}])"
        print(pd.DataFrame(tab).T[conds].to_string())
    print("\nSaved robustness_metrics.csv in", FIG_DIR)

    # ---------- figure 2: performance versus corruption level ----------
    panels = [("blur", "Gaussian blur (sigma, px)", [0, 1.0, 2.0, 3.0]),
              ("brightness", "Brightness factor", [0.5, 1.0, 1.5]),
              ("noise", "Gaussian noise (sigma, 0-255)", [0, 10.0, 20.0, 40.0])]
    def cond_for(kind, lvl):
        if (kind in ("blur", "noise") and lvl == 0) or (kind == "brightness" and lvl == 1.0):
            return "clean"
        return f"{kind}_{float(lvl)}"
    fig, axes = plt.subplots(2, 3, figsize=(7.0, 4.0), sharey="row")
    for r_i, (k, kname) in enumerate([("tail_recall", "Tail-group recall"), ("balanced_acc", "Balanced accuracy")]):
        for c_i, (kind, xlabel, levels) in enumerate(panels):
            ax = axes[r_i, c_i]
            for m in models_r:
                ys, lo, hi = [], [], []
                for lv in levels:
                    cnd = cond_for(kind, lv)
                    a, b = ci(bt[(m, cnd)][k])
                    ys.append(pt[(m, cnd)][k]); lo.append(pt[(m, cnd)][k] - a); hi.append(b - pt[(m, cnd)][k])
                ax.errorbar(levels, ys, yerr=[lo, hi], color=COLOR[m], marker=MARK[m], ms=3, lw=1,
                            capsize=1.5, elinewidth=0.6, label=LABEL[m])
            ax.set_xlabel(xlabel if r_i == 1 else "")
            if c_i == 0:
                ax.set_ylabel(kname)
            ax.grid(alpha=0.3)
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.07))
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "fig2_robustness.png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, "fig2_robustness.pdf"), bbox_inches="tight")
    plt.show()
    print("Saved fig2_robustness.png / .pdf in", FIG_DIR)
print("\nDONE. All tables and figures are in", FIG_DIR)

In [ ]:
#cell 13
import os, glob, numpy as np
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/RareClassRecall_SkinLesion'
PRED_DIR = os.path.join(PROJECT_DIR, 'run', 'preds')
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
TAIL = [class_names.index(c) for c in ['akiec', 'df', 'vasc']]
MEL = class_names.index('mel')
MODELS = ["baseline", "pruned75", "int8", "distilled"]

def metrics(y, pred):
    rec = np.array([(pred[y == c] == c).mean() for c in range(len(class_names))])
    tm = np.isin(y, TAIL); mm = (y == MEL)
    return {"tail_recall": (pred[tm] == y[tm]).mean(), "mel_recall": (pred[mm] == y[mm]).mean(),
            "balanced_acc": rec.mean(), "accuracy": (pred == y).mean()}

res = {}                                   # res[model][seed] = metrics on the TEST set
for m in MODELS:
    for pth in sorted(glob.glob(os.path.join(PRED_DIR, f"{m}_seed*.npz"))):
        sd = int(os.path.basename(pth).split("seed")[1].split(".")[0])
        z = np.load(pth, allow_pickle=True)
        res.setdefault(m, {})[sd] = metrics(z["y_test"], z["test_logits"].argmax(1))
seeds = sorted(res["baseline"])

for k in ["tail_recall", "mel_recall", "balanced_acc", "accuracy"]:
    print(f"\n{k}  (test set, one column per seed;  'diff' = model minus baseline of the SAME seed)")
    print(f"{'model':10s}" + "".join(f"  seed{s:<3d}" for s in seeds) + "    mean   | diff per seed" + " " * 12 + "| seeds with diff < 0")
    for m in [x for x in MODELS if x in res]:
        vals = [res[m][s][k] for s in seeds]
        base = [res["baseline"][s][k] for s in seeds]
        diffs = [v - b for v, b in zip(vals, base)]
        neg = sum(d < 0 for d in diffs)
        dtxt = " ".join(f"{d:+.3f}" for d in diffs) if m != "baseline" else "-"
        ntxt = f"{neg}/{len(seeds)}" if m != "baseline" else "-"
        print(f"{m:10s}" + "".join(f"  {v:.3f} " for v in vals) + f"   {np.mean(vals):.3f}  | {dtxt:24s}| {ntxt}")
print("\nHow to read: if a method is lower than the baseline in ALL seeds, the effect is more believable than if the sign changes between seeds.")

In [ ]:
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt

def corrupt(imgs, kind, level, seed=123):
    """imgs: uint8 (N,H,W,3). Applied on the 0-255 scale, BEFORE normalisation. Values are ROUNDED."""
    if kind == "blur":
        out = np.empty_like(imgs)
        for i in range(len(imgs)):
            out[i] = np.asarray(Image.fromarray(imgs[i]).filter(ImageFilter.GaussianBlur(radius=level)))
        return out
    if kind == "brightness":
        return np.clip(np.rint(imgs.astype(np.float32) * level), 0, 255).astype(np.uint8)
    if kind == "noise":
        rng = np.random.default_rng(seed)
        noisy = imgs.astype(np.float32) + rng.normal(0, level, imgs.shape).astype(np.float32)
        return np.clip(np.rint(noisy), 0, 255).astype(np.uint8)
    raise ValueError(kind)

X_test, y_test = X[te], y[te]

# ---- 1) identity checks: "no corruption" must give exactly the clean images ----
assert (corrupt(X_test[:100], "noise", 0.0) == X_test[:100]).all(), "noise sigma=0 changes the image!"
assert (corrupt(X_test[:100], "brightness", 1.0) == X_test[:100]).all(), "brightness 1.0 changes the image!"
print("blur radius 0 identical to clean:", bool((corrupt(X_test[:100], "blur", 0.0) == X_test[:100]).all()))
d = corrupt(X_test[:300], "noise", 10.0).astype(np.float32) - X_test[:300].astype(np.float32)
print(f"noise sigma=10: std of (noisy - clean) = {d.std():.2f}   (expect about 10)")

# ---- 2) look at the pictures ----
show = [("clean", None), ("blur", 1.0), ("blur", 2.0), ("blur", 3.0), ("brightness", 0.5), ("brightness", 1.5),
        ("noise", 10.0), ("noise", 20.0), ("noise", 40.0)]
fig, axes = plt.subplots(3, len(show), figsize=(18, 6.5))
for r, idx in enumerate([0, 400, 800]):
    for c, (kind, lvl) in enumerate(show):
        img = X_test[idx:idx + 1] if kind == "clean" else corrupt(X_test[idx:idx + 1], kind, lvl)
        axes[r, c].imshow(img[0]); axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title("clean" if kind == "clean" else f"{kind} {lvl}", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "corruption_examples.png"), dpi=300)
plt.show()

# ---- 3) baseline (seed 0) on a few extra levels, to see whether the decline is smooth ----
use_amp = device.type == "cuda"
model = build_resnet50(len(class_names))
model.load_state_dict(torch.load(os.path.join(CKPT_DIR, "baseline_seed0.pt"), map_location="cpu"))
model.to(device)
nv = class_names.index("nv")
specs = [("clean", None), ("noise", 0.0), ("noise", 2.0), ("noise", 5.0), ("noise", 10.0), ("noise", 20.0),
         ("blur", 0.5), ("blur", 1.0), ("blur", 2.0),
         ("brightness", 0.9), ("brightness", 0.7), ("brightness", 0.5)]
print(f"\n{'condition':18s} {'acc':>6s} {'bal_acc':>8s} {'tail_rec':>9s} {'share predicted nv':>19s}")
for kind, lvl in specs:
    imgs = X_test if kind == "clean" else corrupt(X_test, kind, lvl)
    pred = predict_logits(model, imgs, device, use_amp).argmax(1)
    s = summarize(y_test, pred, class_names)
    name = "clean" if kind == "clean" else f"{kind} {lvl}"
    print(f"{name:18s} {s['accuracy']:6.3f} {s['balanced_acc']:8.3f} {s['tail_recall']:9.3f} {(pred == nv).mean():19.3f}")

## Part 2. Seven-fold cross-validation (Cells A, B, C)
Trains and evaluates all six models in every fold. Takes about 105 minutes on a T4 GPU. Safe to re-run: finished files are skipped.

In [ ]:
#cell A
import os, io, time, json, random, shutil, contextlib, glob
import numpy as np, pandas as pd
import torch, torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
from PIL import Image, ImageFilter
from sklearn.model_selection import StratifiedGroupKFold
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/RareClassRecall_SkinLesion'

# ------------------------------ settings ------------------------------
SMOKE_CV = False                       # True = tiny test of fold 0 only (a few minutes). False = the real run.
N_FOLDS = 7
FOLDS_TO_RUN = [0] if SMOKE_CV else list(range(N_FOLDS))
RUN_PRUNE_CONTROL = True              # extra control: fine-tune 5 epochs WITHOUT pruning
EPOCHS, BATCH, LR, WEIGHT_DECAY = 15, 64, 3e-4, 1e-4
FT_EPOCHS, FT_LR, SPARSITY = 5, 1e-4, 0.75
N_CALIB = 500
STUDENT_NAME = "regnet_y_800mf"       # chosen earlier by the fixed size rule
TEMPERATURE, ALPHA = 4.0, 0.5
TAIL_CLASSES, MEL_CLASS = ["akiec", "df", "vasc"], "mel"
# Corruption levels for this run (milder than the first experiment, where every model collapsed):
CORRUPTIONS = [("blur", 0.5), ("blur", 1.0), ("blur", 2.0),
               ("brightness", 0.5), ("brightness", 0.7), ("brightness", 1.5),
               ("noise", 2.0), ("noise", 5.0), ("noise", 10.0)]
if SMOKE_CV:
    epochs_run, ft_epochs_run, batch_run, n_calib_run = 1, 1, 32, 64
else:
    epochs_run, ft_epochs_run, batch_run, n_calib_run = EPOCHS, FT_EPOCHS, BATCH, N_CALIB
CV_DIR = os.path.join(PROJECT_DIR, "cv_smoke" if SMOKE_CV else "cv")
os.makedirs(CV_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| SMOKE_CV =", SMOKE_CV, "| folds to run:", FOLDS_TO_RUN)
if device.type == "cpu" and not SMOKE_CV:
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU.")

MEAN = (0.485, 0.456, 0.406); STD = (0.229, 0.224, 0.225)     # fixed ImageNet numbers


# ------------------------------ helper functions (same as before) ------------------------------
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def per_class_recall(y_true, y_pred, n_classes):
    rec = np.full(n_classes, np.nan)
    for c in range(n_classes):
        m = y_true == c
        if m.sum() > 0:
            rec[c] = float((y_pred[m] == c).mean())
    return rec

def summarize(y_true, y_pred, names):
    rec = per_class_recall(y_true, y_pred, len(names))
    idx = {c: i for i, c in enumerate(names)}
    tail_mask = np.isin(y_true, [idx[c] for c in TAIL_CLASSES])
    mel_mask = y_true == idx[MEL_CLASS]
    return {"accuracy": float((y_true == y_pred).mean()), "balanced_acc": float(np.nanmean(rec)),
            "tail_recall": float((y_pred[tail_mask] == y_true[tail_mask]).mean()) if tail_mask.any() else float("nan"),
            "mel_recall": float((y_pred[mel_mask] == y_true[mel_mask]).mean()) if mel_mask.any() else float("nan")}

def autocast(dev, use_amp):
    if not use_amp:
        return contextlib.nullcontext()
    return torch.autocast(device_type=dev.type, dtype=torch.float16)

def make_scaler(use_amp):
    try:
        return torch.amp.GradScaler("cuda", enabled=use_amp)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=use_amp)

def prep(xb_uint8):
    x = xb_uint8.permute(0, 3, 1, 2).float() / 255.0
    mean = torch.tensor(MEAN, device=x.device).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=x.device).view(1, 3, 1, 1)
    return (x - mean) / std

def augment(x):
    b = x.size(0)
    fh = torch.rand(b, device=x.device) < 0.5
    x = torch.where(fh[:, None, None, None], torch.flip(x, dims=[3]), x)
    fv = torch.rand(b, device=x.device) < 0.5
    x = torch.where(fv[:, None, None, None], torch.flip(x, dims=[2]), x)
    k = random.randint(0, 3)
    if k:
        x = torch.rot90(x, k, dims=[2, 3])
    return x

def build_resnet50(n_classes, pretrained=False):
    if pretrained:
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
    else:
        m = torchvision.models.resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    return m

def replace_head(m, n):
    if hasattr(m, "fc") and isinstance(m.fc, nn.Linear):
        m.fc = nn.Linear(m.fc.in_features, n)
    elif hasattr(m, "classifier") and isinstance(m.classifier, nn.Sequential) and isinstance(m.classifier[-1], nn.Linear):
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n)
    else:
        raise RuntimeError("unknown classifier-head layout")
    return m

def make_student(name, n, pretrained):
    ctor = getattr(torchvision.models, name)
    return replace_head(ctor(weights="DEFAULT") if pretrained else ctor(weights=None), n)

@torch.no_grad()
def predict_logits(model, X_np, dev, use_amp, bs=128):
    model.eval()
    outs = []
    for i in range(0, len(X_np), bs):
        xb = prep(torch.from_numpy(X_np[i:i + bs]).to(dev))
        with autocast(dev, use_amp):
            out = model(xb)
        outs.append(out.float().cpu())
    return torch.cat(outs).numpy()

def state_size_mb(model):
    buf = io.BytesIO(); torch.save(model.state_dict(), buf)
    return buf.tell() / 1e6

def count_nonzero_params(model):
    return int(sum((p != 0).sum().item() for p in model.parameters()))

def pick_smoke_subset(role, yy, which, per_class):
    ids = np.where(role == which)[0]
    keep = []
    for c in np.unique(yy[ids]):
        keep.extend(ids[yy[ids] == c][:per_class].tolist())
    return np.array(sorted(keep))

def corrupt(imgs, kind, level, seed=123):
    """imgs: uint8 (N,H,W,3). Applied on the 0-255 scale BEFORE normalisation. Values are rounded."""
    if kind == "blur":
        out = np.empty_like(imgs)
        for i in range(len(imgs)):
            out[i] = np.asarray(Image.fromarray(imgs[i]).filter(ImageFilter.GaussianBlur(radius=level)))
        return out
    if kind == "brightness":
        return np.clip(np.rint(imgs.astype(np.float32) * level), 0, 255).astype(np.uint8)
    if kind == "noise":
        rng = np.random.default_rng(seed)
        noisy = imgs.astype(np.float32) + rng.normal(0, level, imgs.shape).astype(np.float32)
        return np.clip(np.rint(noisy), 0, 255).astype(np.uint8)
    raise ValueError(kind)


# ------------------------------ data ------------------------------
SIZE = 128
NPZ_LOCAL = f"/content/ham10000_{SIZE}.npz"
NPZ_DRIVE = os.path.join(PROJECT_DIR, f"ham10000_{SIZE}.npz")
if not os.path.exists(NPZ_LOCAL):
    shutil.copy(NPZ_DRIVE, NPZ_LOCAL)
d = np.load(NPZ_LOCAL)
X, y, lesion, image_id, split_saved = d["X"], d["y"], d["lesion"], d["image_id"], d["split"]
class_names = [str(c) for c in d["class_names"]]
n_classes = len(class_names)
print("images:", X.shape, "| classes:", class_names)

# ------------------------------ folds (7 folds, same recipe as the first split) ------------------------------
meta = pd.read_csv(os.path.join(PROJECT_DIR, "HAM10000_metadata.csv")).reset_index(drop=True)
assert (meta["image_id"].values == image_id).all(), "metadata order differs from the cached images"
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
fold_of = np.full(len(meta), -1)
for k, (_, t) in enumerate(sgkf.split(meta, meta["dx"], groups=meta["lesion_id"])):
    fold_of[t] = k
assert (fold_of >= 0).all()
# Fold 0 = test and fold 1 = validation must reproduce the split used in the first experiment:
assert ((fold_of == 0) == (split_saved == "test")).all() and ((fold_of == 1) == (split_saved == "val")).all(), \
    "folds do not reproduce the first split (different scikit-learn version?). Stop and check the scikit-learn version (see README)."
print("Fold check passed: fold 0 / fold 1 reproduce the first test / validation sets.")

def fold_indices(k):
    role = np.full(len(y), "train", dtype="<U5")
    role[fold_of == (k + 1) % N_FOLDS] = "val"
    role[fold_of == k] = "test"
    if SMOKE_CV:
        return (pick_smoke_subset(role, y, "train", 40), pick_smoke_subset(role, y, "val", 20),
                pick_smoke_subset(role, y, "test", 20))
    return np.where(role == "train")[0], np.where(role == "val")[0], np.where(role == "test")[0]

tail_ids = [class_names.index(c) for c in TAIL_CLASSES]
print("\nfold | test images | tail images in test | df | vasc | melanoma")
for k in range(N_FOLDS):
    m = fold_of == k
    print(f"  {k}  | {m.sum():5d}       | {np.isin(y[m], tail_ids).sum():4d}                | "
          f"{(y[m] == class_names.index('df')).sum():3d}| {(y[m] == class_names.index('vasc')).sum():4d} | "
          f"{(y[m] == class_names.index('mel')).sum():4d}")
print("\nAcross all folds every image is a test image exactly once.")

In [ ]:
#cell B
def train_classifier(model, tr_idx, va_idx, epochs, lr, teacher=None, tag=""):
    """Train with plain cross-entropy (teacher=None) or with distillation (teacher given).
    The best epoch is chosen on the VALIDATION fold only."""
    use_amp = device.type == "cuda"
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)   # created AFTER pruning (if any)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = make_scaler(use_amp)
    ce = nn.CrossEntropyLoss()                                    # plain, no class weights
    Xtr_d = torch.from_numpy(X[tr_idx]).to(device); ytr_d = torch.from_numpy(y[tr_idx]).to(device)
    n = len(Xtr_d); T = TEMPERATURE
    best_score, best_epoch, best_state = -1.0, -1, None
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_run):
            idx = perm[i:i + batch_run]
            if len(idx) < 2:
                continue
            xb = augment(prep(Xtr_d[idx])); yb = ytr_d[idx]
            if teacher is not None:
                with torch.no_grad():
                    with autocast(device, use_amp):
                        t_logits = teacher(xb)
            with autocast(device, use_amp):
                s_logits = model(xb)
            s32 = s_logits.float()
            if teacher is None:
                loss = ce(s32, yb)
            else:
                kl = F.kl_div(F.log_softmax(s32 / T, dim=1), F.softmax(t_logits.float() / T, dim=1),
                              reduction="batchmean") * (T * T)
                loss = ALPHA * kl + (1 - ALPHA) * ce(s32, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        va_pred = predict_logits(model, X[va_idx], device, use_amp).argmax(1)
        s = summarize(y[va_idx], va_pred, class_names)
        if s["balanced_acc"] > best_score:
            best_score, best_epoch = s["balanced_acc"], epoch + 1
            best_state = {kk: v.detach().cpu().clone() for kk, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f"    [{tag}] best epoch {best_epoch}/{epochs}, validation balanced accuracy {best_score:.3f}, "
          f"{time.time() - t0:.0f}s")
    del Xtr_d, ytr_d
    return model, best_epoch, best_score


def prunable(model):
    return [(m, "weight") for m in model.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]


def prune_stage(base_state, tr_idx, va_idx, amount):
    """amount > 0: global L1 magnitude pruning, then fine-tune. amount = 0: fine-tune only (control)."""
    m = build_resnet50(n_classes)
    m.load_state_dict(base_state)
    m.to(device)
    if amount > 0:
        prune.global_unstructured(prunable(m), pruning_method=prune.L1Unstructured, amount=amount)
    m, be, bs = train_classifier(m, tr_idx, va_idx, ft_epochs_run, FT_LR, tag=f"prune {amount:.2f}")
    if amount > 0:
        for mod, name in prunable(m):                 # make the zeros permanent
            try:
                prune.remove(mod, name)
            except ValueError:
                pass
    tot = zer = 0
    for mod, _ in prunable(m):
        tot += mod.weight.numel(); zer += int((mod.weight == 0).sum().item())
    return m, dict(best_epoch=be, val_bal_acc=bs, sparsity=zer / tot)


def quantize_int8(base_state, calib_X):
    from torchvision.models.quantization import resnet50 as q_resnet50
    torch.backends.quantized.engine = "fbgemm"
    m = q_resnet50(weights=None, quantize=False)
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    m.load_state_dict(base_state)
    m.eval()
    m.fuse_model()
    m.qconfig = torch.ao.quantization.get_default_qconfig("fbgemm")
    torch.ao.quantization.prepare(m, inplace=True)
    with torch.no_grad():
        for i in range(0, len(calib_X), 64):
            m(prep(torch.from_numpy(calib_X[i:i + 64])))
    torch.ao.quantization.convert(m, inplace=True)
    return m


def eval_conditions(model, dev, amp, corr_sets):
    return {key: predict_logits(model, arr, dev, amp) for key, arr in corr_sets.items()}

print("functions defined.")

In [ ]:
#cell C
cpu = torch.device("cpu")
use_amp = device.type == "cuda"
cond_key = lambda kind, lvl: f"{kind}_{lvl}"
MODEL_NAMES = ["baseline", "pruned75", "int8", "student_ce", "student_kd"] + (["finetune_only"] if RUN_PRUNE_CONTROL else [])
exists = lambda p: os.path.exists(p) and not SMOKE_CV
out_path = lambda k, name: os.path.join(CV_DIR, f"fold{k}", f"{name}.npz")

print("progress so far (x = finished):")
for k in range(N_FOLDS):
    print(f"  fold {k}: " + "  ".join(f"{m}:{'x' if os.path.exists(out_path(k, m)) else '-'}" for m in MODEL_NAMES))
print()

t_all = time.time()
for k in FOLDS_TO_RUN:
    fold_dir = os.path.join(CV_DIR, f"fold{k}"); os.makedirs(fold_dir, exist_ok=True)
    tr_idx, va_idx, te_idx = fold_indices(k)
    todo = [m for m in MODEL_NAMES if not exists(out_path(k, m))]
    if not todo:
        print(f"fold {k}: everything done, skipping"); continue
    print(f"===== fold {k}: train {len(tr_idx)} | validation {len(va_idx)} | test {len(te_idx)} | to do: {todo} =====")
    t_fold = time.time()
    np.savez_compressed(os.path.join(fold_dir, "meta.npz"), te_idx=te_idx, image_id=image_id[te_idx],
                        lesion=lesion[te_idx], y=y[te_idx], class_names=np.array(class_names))

    # the corrupted test images of this fold (the SAME for every model)
    X_te = X[te_idx]
    corr = {"clean": X_te}
    for kind, lvl in CORRUPTIONS:
        corr[cond_key(kind, lvl)] = corrupt(X_te, kind, lvl)

    def finish(name, model, dev, amp, extra, t0):
        logits = eval_conditions(model, dev, amp, corr)
        np.savez_compressed(out_path(k, name), **logits, **extra)
        acc = float((logits["clean"].argmax(1) == y[te_idx]).mean())
        print(f"    [{name}] saved. (clean test accuracy {acc:.3f}; total {time.time() - t0:.0f}s)")

    def free():
        if device.type == "cuda":
            torch.cuda.empty_cache()

    # ---- baseline (every other model needs it) ----
    bpath = os.path.join(fold_dir, "baseline.pt")
    if "baseline" in todo:
        t0 = time.time(); set_seed(k)
        m = build_resnet50(n_classes, pretrained=True)
        m, be, bs = train_classifier(m, tr_idx, va_idx, epochs_run, LR, tag="baseline")
        base_state = {kk: v.cpu() for kk, v in m.state_dict().items()}
        torch.save(base_state, bpath)
        finish("baseline", m, device, use_amp,
               dict(size_mb=state_size_mb(m), nonzero_params=count_nonzero_params(m), best_epoch=be, val_bal_acc=bs), t0)
        del m; free()
    else:
        base_state = torch.load(bpath, map_location="cpu")

    # ---- pruning (75%) and the fine-tune-only control ----
    for name, amount in [("pruned75", SPARSITY), ("finetune_only", 0.0)]:
        if name in todo:
            t0 = time.time(); set_seed(k)
            m, info = prune_stage(base_state, tr_idx, va_idx, amount)
            finish(name, m, device, use_amp,
                   dict(size_mb=state_size_mb(m), nonzero_params=count_nonzero_params(m), **info), t0)
            del m; free()

    # ---- INT8 post-training quantization (runs on the CPU) ----
    if "int8" in todo:
        t0 = time.time()
        calib_idx = np.random.default_rng(k).choice(tr_idx, size=min(n_calib_run, len(tr_idx)), replace=False)
        q = quantize_int8(base_state, X[calib_idx])
        finish("int8", q, cpu, False, dict(size_mb=state_size_mb(q)), t0)
        del q

    # ---- student trained with plain cross-entropy (control) ----
    if "student_ce" in todo:
        t0 = time.time(); set_seed(k)
        s = make_student(STUDENT_NAME, n_classes, pretrained=True)
        s, be, bs = train_classifier(s, tr_idx, va_idx, epochs_run, LR, tag="student, plain cross-entropy")
        finish("student_ce", s, device, use_amp,
               dict(size_mb=state_size_mb(s), nonzero_params=count_nonzero_params(s), best_epoch=be, val_bal_acc=bs), t0)
        del s; free()

    # ---- student trained by distillation from the baseline of this fold ----
    if "student_kd" in todo:
        t0 = time.time()
        teacher = build_resnet50(n_classes); teacher.load_state_dict(base_state); teacher.to(device).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        set_seed(k)
        s = make_student(STUDENT_NAME, n_classes, pretrained=True)
        s, be, bs = train_classifier(s, tr_idx, va_idx, epochs_run, LR, teacher=teacher, tag="student, distillation")
        finish("student_kd", s, device, use_amp,
               dict(size_mb=state_size_mb(s), nonzero_params=count_nonzero_params(s), best_epoch=be, val_bal_acc=bs), t0)
        del s, teacher; free()

    del corr, X_te
    print(f"===== fold {k} finished in {(time.time() - t_fold) / 60:.1f} min =====\n")

print(f"DONE for folds {FOLDS_TO_RUN}. Total {(time.time() - t_all) / 60:.1f} min. Files are in {CV_DIR}")

## Part 3. Analysis, tables, figure (Cells D and E)

In [ ]:
#cell D  (analysis: pooled 7-fold, lesion bootstrap, all comparisons)
import os, glob, itertools
import numpy as np, pandas as pd
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass

if 'PROJECT_DIR' not in globals():
    PROJECT_DIR = '/content/drive/MyDrive/RareClassRecall_SkinLesion'
CV_DIR  = os.path.join(PROJECT_DIR, "cv")
OUT_DIR = os.path.join(CV_DIR, "results"); os.makedirs(OUT_DIR, exist_ok=True)

MODELS  = ["baseline", "pruned75", "finetune_only", "int8", "student_ce", "student_kd"]
TAIL_CLASSES, MEL_CLASS = ["akiec", "df", "vasc"], "mel"
N_BOOT = 1000
# (a, b) = a minus b.  PRIMARY first, then controls.
COMPARISONS = [("pruned75", "baseline"), ("int8", "baseline"), ("student_kd", "baseline"),
               ("pruned75", "finetune_only"), ("student_kd", "student_ce"),
               ("finetune_only", "baseline"), ("student_ce", "baseline")]
METRICS = ["tail_recall", "mel_recall", "balanced_acc", "accuracy"]

# ---------------- load every fold ----------------
fold_dirs = sorted(glob.glob(os.path.join(CV_DIR, "fold*")), key=lambda p: int(p.split("fold")[-1]))
fold_dirs = [d for d in fold_dirs if os.path.exists(os.path.join(d, "meta.npz"))]
assert fold_dirs, "no fold folders found"
y, lesion, fold_id = [], [], []
logits, info = {}, {}                       # logits[model][cond] = list over folds
for d in fold_dirs:
    k = int(d.split("fold")[-1])
    mt = np.load(os.path.join(d, "meta.npz"), allow_pickle=True)
    y.append(mt["y"]); lesion.append(mt["lesion"]); fold_id.append(np.full(len(mt["y"]), k))
    class_names = [str(c) for c in mt["class_names"]]
    for m in MODELS:
        z = np.load(os.path.join(d, f"{m}.npz"), allow_pickle=True)
        for key in z.files:
            if key in ("size_mb", "nonzero_params", "best_epoch", "val_bal_acc"):
                info.setdefault(m, {}).setdefault(key, []).append(float(z[key]))
            else:
                logits.setdefault(m, {}).setdefault(key, []).append(z[key])
y = np.concatenate(y); lesion = np.concatenate(lesion); fold_id = np.concatenate(fold_id)
folds = sorted(set(fold_id.tolist()))
conds = list(logits["baseline"].keys())
pred = {m: {c: np.concatenate(logits[m][c]).argmax(1) for c in conds} for m in MODELS}
n, C = len(y), len(class_names)
TAIL = [class_names.index(c) for c in TAIL_CLASSES]; MEL = class_names.index(MEL_CLASS)
print(f"{len(folds)} folds, {n} test images in total, {len(np.unique(lesion))} lesions, {len(conds)} conditions")
assert len(np.unique(lesion)) == len(set(lesion.tolist()))
for k in folds:   # a lesion must be in exactly one fold
    pass
assert (pd.Series(fold_id).groupby(lesion).nunique() == 1).all(), "a lesion appears in two folds!"

# ---------------- lesion bootstrap (same weights for ALL models -> paired) ----------------
uniq, inv = np.unique(lesion, return_inverse=True)
L = len(uniq); rng = np.random.default_rng(0)
cnt = np.stack([np.bincount(rng.integers(0, L, L), minlength=L) for _ in range(N_BOOT)]).astype(float)
W = np.vstack([np.ones((1, n)), cnt[:, inv]])          # row 0 = the real data (point estimate)
onehot = np.eye(C)[y]
DEN = W @ onehot                                        # (1+N_BOOT, C)

def metrics_from(p):
    NUM = W @ (onehot * (p == y)[:, None])
    with np.errstate(divide="ignore", invalid="ignore"):
        rec = np.where(DEN > 0, NUM / DEN, np.nan)
    return {"tail_recall": NUM[:, TAIL].sum(1) / DEN[:, TAIL].sum(1),
            "mel_recall": NUM[:, MEL] / DEN[:, MEL],
            "balanced_acc": np.nanmean(rec, axis=1),
            "accuracy": NUM.sum(1) / DEN.sum(1)}

B = {m: {c: metrics_from(pred[m][c]) for c in conds} for m in MODELS}   # B[m][c][metric] = array(1+N_BOOT)
ci = lambda a: (float(np.percentile(a[1:], 2.5)), float(np.percentile(a[1:], 97.5)))

# ---------------- 1) main table (clean) ----------------
rows = []
for m in MODELS:
    for k_ in METRICS:
        a = B[m]["clean"][k_]; lo, hi = ci(a)
        rows.append(dict(model=m, metric=k_, value=a[0], ci_low=lo, ci_high=hi))
main = pd.DataFrame(rows); main.to_csv(os.path.join(OUT_DIR, "main_metrics.csv"), index=False)
print("\n=== CLEAN test, all folds pooled (95% CI, lesion bootstrap) ===")
print(main.pivot(index="model", columns="metric", values="value").loc[MODELS, METRICS].round(3).to_string())

# ---------------- 2) sizes ----------------
sz = pd.DataFrame([dict(model=m, size_mb=np.mean(info[m]["size_mb"]), nonzero_params_M=np.mean(info[m]["nonzero_params"]) / 1e6 if "nonzero_params" in info[m] else np.nan)
                   for m in MODELS])
sz["x_smaller_size"] = sz.size_mb.iloc[0] / sz.size_mb
sz["x_smaller_nonzero"] = sz.nonzero_params_M.iloc[0] / sz.nonzero_params_M
sz.to_csv(os.path.join(OUT_DIR, "sizes.csv"), index=False); print("\n=== SIZES ===\n", sz.round(2).to_string(index=False))

# ---------------- 3) paired differences, clean + every corruption ----------------
rows = []
for a, b in COMPARISONS:
    for c in conds:
        for k_ in METRICS:
            d = B[a][c][k_] - B[b][c][k_]; lo, hi = ci(d)
            rows.append(dict(a=a, b=b, cond=c, metric=k_, diff=d[0], ci_low=lo, ci_high=hi,
                             excludes_zero=bool(lo > 0 or hi < 0)))
comp = pd.DataFrame(rows); comp.to_csv(os.path.join(OUT_DIR, "comparisons.csv"), index=False)
print("\n=== PAIRED DIFFERENCES on clean images (a minus b) ===")
print(comp[comp.cond == "clean"].pivot_table(index=["a", "b"], columns="metric", values="diff", sort=False).round(3).to_string())
print("\nCI that exclude zero (clean):")
print(comp[(comp.cond == "clean") & comp.excludes_zero][["a", "b", "metric", "diff", "ci_low", "ci_high"]].round(3).to_string(index=False))
print("(7 comparisons x 4 metrics = 28 tests: a few will exclude zero by chance. Say so in the paper.)")

# ---------------- 4) fold-by-fold (clean): does the sign agree across folds? ----------------
rows = []
for k in folds:
    msk = fold_id == k
    def fm(m):
        p, yy = pred[m]["clean"][msk], y[msk]
        rec = np.array([(p[yy == c] == c).mean() if (yy == c).any() else np.nan for c in range(C)])
        tm = np.isin(yy, TAIL)
        return dict(tail_recall=(p[tm] == yy[tm]).mean(), balanced_acc=np.nanmean(rec), accuracy=(p == yy).mean())
    base = {m: fm(m) for m in MODELS}
    for m in MODELS:
        rows.append(dict(fold=k, model=m, **base[m]))
fold_df = pd.DataFrame(rows); fold_df.to_csv(os.path.join(OUT_DIR, "per_fold_clean.csv"), index=False)
print("\n=== per-fold TAIL RECALL (clean) ===")
tab = fold_df.pivot(index="fold", columns="model", values="tail_recall")[MODELS]; print(tab.round(3).to_string())
print("\nfolds where (a - b) < 0 for tail recall, out of", len(folds))
for a, b in COMPARISONS:
    dd = tab[a] - tab[b]
    print(f"  {a:14s} - {b:14s}: mean {dd.mean():+.3f} | negative in {(dd < 0).sum()}/{len(folds)} folds | range {dd.min():+.3f} to {dd.max():+.3f}")
print("(Folds share training data, so they are NOT independent: use this as a consistency check, not as a test.)")

# ---------------- 5) collapse check under corruption ----------------
print("\n=== models that answer one class for almost everything (balanced acc < 0.17) ===")
flag = False
for m in MODELS:
    for c in conds:
        if B[m][c]["balanced_acc"][0] < 0.17:
            top = np.bincount(pred[m][c], minlength=C).max() / n
            print(f"  {m:14s} {c:14s} bal_acc {B[m][c]['balanced_acc'][0]:.3f} | share of one class {top:.2f}"); flag = True
if not flag: print("  none")

# ---------------- 6) robustness table (tail recall + bal acc) ----------------
rob = pd.DataFrame([dict(model=m, cond=c, metric=k_, value=B[m][c][k_][0], ci_low=ci(B[m][c][k_])[0], ci_high=ci(B[m][c][k_])[1])
                    for m in MODELS for c in conds for k_ in METRICS])
rob.to_csv(os.path.join(OUT_DIR, "robust_metrics.csv"), index=False)
print("\n=== TAIL RECALL by condition ===")
print(rob[rob.metric == "tail_recall"].pivot(index="cond", columns="model", values="value").loc[conds, MODELS].round(3).to_string())
print("\nSaved CSVs in", OUT_DIR)

In [ ]:
#cell E  (tables + robustness figure; vector PDF + 300-dpi PNG; TrueType fonts)
import os, numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt

if 'PROJECT_DIR' not in globals():
    PROJECT_DIR = '/content/drive/MyDrive/RareClassRecall_SkinLesion'
RES_DIR = os.path.join(PROJECT_DIR, "cv", "results")
FIG_DIR = os.path.join(PROJECT_DIR, "cv", "figures"); os.makedirs(FIG_DIR, exist_ok=True)

# ---- journal-safe fonts: TrueType (Type 42), no Type 3 ----
plt.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9,
    "mathtext.fontset": "stix",
    "axes.linewidth": 0.6, "lines.linewidth": 1.1, "lines.markersize": 3.5,
    "savefig.dpi": 300, "figure.dpi": 150,
})

main = pd.read_csv(os.path.join(RES_DIR, "main_metrics.csv"))
sizes = pd.read_csv(os.path.join(RES_DIR, "sizes.csv")).set_index("model")
comp = pd.read_csv(os.path.join(RES_DIR, "comparisons.csv"))
rob = pd.read_csv(os.path.join(RES_DIR, "robust_metrics.csv"))

NAME = {"baseline": "Baseline ResNet-50", "pruned75": "Pruned 75%", "finetune_only": "Fine-tune only (control)",
        "int8": "INT8 (post-training)", "student_ce": "Student, plain CE (control)", "student_kd": "Student, distilled"}
MODELS = list(NAME)
tex = lambda s: s.replace("%", r"\%")                       # LaTeX: % must be escaped
f2 = lambda v: "0.00" if abs(v) < 0.005 else f"{v:.2f}".replace("-", "$-$")   # 2 decimals; no "-0.00"

# ================= TABLE 1: clean results + size =================
SHORT1 = {"baseline": "Baseline", "pruned75": "Pruned 75\\%", "finetune_only": "Fine-tune only", "int8": "INT8 (PTQ)",
          "student_ce": "Student (CE)", "student_kd": "Student (KD)"}
rows = []
for m in MODELS:
    g = lambda k: main[(main.model == m) & (main.metric == k)].value.iloc[0]
    rows.append([SHORT1[m], f"{sizes.loc[m, 'size_mb']:.1f}", f2(g("tail_recall")), f2(g("mel_recall")), f2(g("balanced_acc")), f2(g("accuracy"))])
t1 = ["\\begin{table}[t]", "\\centering\\setlength{\\tabcolsep}{3pt}",
      "\\caption{Clean test results, all seven folds pooled (10,015 images). MB: size of the saved file. Tail: tail-group recall. Mel: melanoma recall. Bal.: balanced accuracy. CE: student trained without a teacher (control). KD: distilled student.}",
      "\\label{tab:main}", "\\begin{tabular}{lrcccc}", "\\hline",
      "Model & MB & Tail & Mel & Bal. & Acc. \\\\", "\\hline"]
t1 += [" & ".join(r) + " \\\\" for r in rows]
t1 += ["\\hline", "\\end{tabular}", "\\end{table}"]
open(os.path.join(FIG_DIR, "table1_main.tex"), "w").write("\n".join(t1))

# ================= TABLE 2: paired differences (a minus b), bold if the 95% CI excludes 0 =================
PAIRS = [("pruned75", "baseline"), ("pruned75", "finetune_only"), ("int8", "baseline"),
         ("student_kd", "baseline"), ("student_kd", "student_ce"), ("student_ce", "baseline")]
short = {"baseline": "base", "pruned75": "pruned", "finetune_only": "FT", "int8": "INT8", "student_ce": "CE", "student_kd": "KD"}
def cell(a, b, metric, cond="clean"):
    r = comp[(comp.a == a) & (comp.b == b) & (comp.cond == cond) & (comp.metric == metric)].iloc[0]
    s = f"{f2(r['diff'])} [{f2(r['ci_low'])},{f2(r['ci_high'])}]"
    return f"\\textbf{{{s}}}" if r["excludes_zero"] else s
t2 = ["\\begin{table}[t]", "\\centering\\setlength{\\tabcolsep}{2pt}",
      "\\caption{Paired differences on clean images (first minus second) with 95\\% lesion-bootstrap intervals. Bold: interval excludes zero. FT: fine-tune only. CE, KD: students without and with distillation.}",
      "\\label{tab:diff}", "\\begin{tabular}{lcc}", "\\hline", "Comparison & Tail recall & Bal. accuracy \\\\", "\\hline"]
for a, b in PAIRS:
    t2.append(f"{short[a]} vs {short[b]} & {cell(a, b, 'tail_recall')} & {cell(a, b, 'balanced_acc')} \\\\")
t2 += ["\\hline", "\\end{tabular}", "\\end{table}"]
open(os.path.join(FIG_DIR, "table2_diff.tex"), "w").write("\n".join(t2))
print("\n".join(t1)); print(); print("\n".join(t2))

# ================= FIGURE 1: tail recall under corruption (3 panels) =================
STYLE = {"baseline": ("#222222", "o", "-"), "pruned75": ("#1f77b4", "s", "-"), "int8": ("#d62728", "^", "-"),
         "student_kd": ("#2ca02c", "D", "-"), "finetune_only": ("#8c8c8c", "x", "--"), "student_ce": ("#b0b0b0", "+", "--")}
panels = [("blur", r"Blur $\sigma$ (pixels)", 0.0), ("brightness", "Brightness factor", 1.0), ("noise", r"Noise $\sigma$ (0$-$255)", 0.0)]
LEG = {"baseline": "Baseline", "pruned75": "Pruned 75%", "int8": "INT8 (PTQ)", "student_kd": "Student (KD)", "finetune_only": "Fine-tune only", "student_ce": "Student (CE)"}
tr = rob[rob.metric == "tail_recall"]
fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.5), sharey=True, constrained_layout=True)
for ax, (kind, xlabel, clean_x) in zip(axes, panels):
    levels = sorted({float(c.split("_")[1]) for c in tr.cond.unique() if c.startswith(kind + "_")} | {clean_x})
    def vals(m, col="value"):
        out = []
        for lv in levels:
            cond = "clean" if lv == clean_x else f"{kind}_{lv}"
            out.append(tr[(tr.model == m) & (tr.cond == cond)][col].iloc[0])
        return out
    ax.fill_between(levels, vals("baseline", "ci_low"), vals("baseline", "ci_high"), color="#222222", alpha=0.12, linewidth=0)
    for m in ["baseline", "pruned75", "int8", "student_kd", "finetune_only", "student_ce"]:
        c, mk, ls = STYLE[m]
        ax.plot(levels, vals(m), color=c, marker=mk, linestyle=ls, label=LEG[m])
    ax.set_xlabel(xlabel); ax.set_xticks(levels); ax.set_xticklabels([f"{v:g}" for v in levels]); ax.set_ylim(0, 0.8)
    ax.grid(alpha=0.25, linewidth=0.4)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
axes[0].set_ylabel("Tail-group recall")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="outside lower center", ncol=3, frameon=False)
pdf_path = os.path.join(FIG_DIR, "fig1_robustness.pdf"); png_path = os.path.join(FIG_DIR, "fig1_robustness.png")
fig.savefig(pdf_path)                       # vector
fig.savefig(png_path, dpi=300)              # raster, 300 dpi
plt.show()
print(f"\nSaved {pdf_path} (vector) and {png_path} (300 dpi). Figure size: {fig.get_size_inches()} inches = full text width of the ISBI page.")

# check: fonts really embedded as TrueType (Type 42), no Type 3
raw = open(pdf_path, "rb").read()
print("Type 3 fonts in PDF:", b"/Type3" in raw, "(must be False)")
from PIL import Image
im = Image.open(png_path); print("PNG pixels:", im.size, "-> dpi:", im.info.get("dpi"))

# ================= numbers needed for the Results text =================
print("\n=== TAIL RECALL: paired difference vs baseline under each corruption (95% CI; * = excludes 0) ===")
q = comp[(comp.metric == "tail_recall") & (comp.b == "baseline") & (comp.a.isin(["pruned75", "int8", "student_kd", "student_ce"]))]
for c in q.cond.unique():
    s = "  ".join(f"{a}: {r['diff']:+.2f} [{r['ci_low']:+.2f},{r['ci_high']:+.2f}]{'*' if r['excludes_zero'] else ''}"
                  for a in ["pruned75", "int8", "student_kd", "student_ce"] for r in [q[(q.cond == c) & (q.a == a)].iloc[0]])
    print(f"{c:15s} {s}")
print("\n=== bootstrap CI of every main number (clean) ===")
print(main.assign(txt=lambda d: d.apply(lambda r: f"{r.value:.2f} [{r.ci_low:.2f},{r.ci_high:.2f}]", axis=1))
      .pivot(index="model", columns="metric", values="txt").loc[MODELS].to_string())
print("\n=== all differences for the text (clean): diff [CI] ===")
for a, b in [("pruned75", "baseline"), ("pruned75", "finetune_only"), ("int8", "baseline"), ("student_kd", "baseline"), ("student_kd", "student_ce"), ("student_ce", "baseline")]:
    print(f"{a} - {b}: " + " | ".join(f"{k} {cell(a, b, k).replace('$-$','-').replace(chr(92)+'textbf','')}" for k in ["tail_recall", "mel_recall", "balanced_acc", "accuracy"]))